# ICT Model — USD/JPY (Inner Circle Trader)

شغّل الخلية دي بالكامل لتدريب كل الفريمات (5min → 1W). النتائج (الموديلات + المقاييس) هتتحفظ في `models/ict_usdjpy/`.

In [2]:
# ============================================================
# usdjpy_ict_model.py  —  USD/JPY  |  ICT (Inner Circle Trader) School — USD/JPY Forex
# ============================================================
# نسخة مُطبَّقة على الفضة (XAG/USD) من نفس موديل الدهب (gold_ict_model.py)،
# بنفس الأركتيكتشر الأساسي، لكن مع تظبيطات خاصة بطبيعة الفضة:
#   🔧 السبريد/السليباج في الباك تيست تم تكبيرهم (USD/JPY سيولة OTC أرفع
#      نسبياً من الدهب، فالسبريد النسبي بالـ bps أعلى).
#   🔧 مضاعفات ATR (SL/TP) والـ confidence_threshold اتزودوا شوية —
#      تقلب USD/JPY النسبي (ATR% للسعر) أعلى من الدهب عادة، فبيحتاج
#      مساحة أوسع للستوب وفلترة أقوى للضجيج خصوصاً على الفريمات الصغيرة.
#   🆕 فيتشور جديد: نسبة الدهب/الفضة (Gold/Silver Ratio) وانحرافها —
#      مؤشر كلاسيكي في تداول المعادن النفيسة مالوش معنى في موديل الدهب
#      نفسه (مينفعش يقيس نفسه بنفسه) لكنه بيضيف قيمة حقيقية هنا.
# ============================================================
#
# نفس الارتكتشر المستخدم في موديلي "Technical/Advanced" و
# "Classic/Harmonic/Elliott" لكن مبني بالكامل على مفاهيم ICT:
#
#   ✅ Market Structure (BOS / CHoCH) على Swing Points حقيقية
#   ✅ Order Blocks (Bullish / Bearish) + Mitigation tracking
#   ✅ Fair Value Gaps (FVG) + نسبة الامتلاء
#   ✅ Liquidity: Equal Highs/Lows + Liquidity Sweeps (Stop Hunts)
#   ✅ Premium / Discount + OTE (Optimal Trade Entry 62%-79%)
#   ✅ Displacement candles (حركة مؤسساتية قوية)
#   ✅ Breaker Blocks (OB فاشل تحول لدعم/مقاومة)
#   ✅ Kill Zones + Session Ranges (Asia / London / NY / Silver Bullet)
#   ✅ Multi-Timeframe Bias (HTF market structure + premium/discount)
#   ✅ نفس الـ Timeframes المدعومة في الموديلين التانيين:
#      5min, 15min, 45min, 1h, 2h, 4h, 1d, 1W
#
# ── إصلاحات جوهرية عن النسختين السابقتين ──────────────────────
#   🔧 DATA LEAKAGE (كان بيأثر على الموديلين): الـ "final model" كان
#      بيتدرب على 100% من الداتا (شامل آخر 15-20% اللي بيتـ"قيّم" عليها
#      كـ holdout) فالنتائج كانت وهمية. هنا الـ holdout معزول تماماً
#      من أول التدريب، والموديل اللي بيتقيّم عليه لم يرَ هذه البيانات
#      إطلاقاً. موديل الإنتاج النهائي (اللي بيتدرب على 100%) بيتحفظ
#      منفصل ومُعلَّم بوضوح إنه غير مقيّم عليه هولد أوت حقيقي.
#   🔧 API KEY: بقى بيتقرأ من environment variable مش hardcoded.
#   🔧 سحب الداتا: كل فريم بيتسحب مباشرة (native interval) من الـ API
#      مش بالـ resample من 5min فقط — عشان الفريمات الكبيرة (1d, 1W)
#      تاخد تاريخ كافي فعلاً (10-15 سنة) مش أشهر معدودة.
#   🔧 BACKTESTER كان LONG-ONLY: إشارات SELL كانت بتتحسب لكن محدش كان
#      بيفتح بيها صفقة SHORT فعلياً — نص الفرص كانت بتتضيع. اتصلح.
# ============================================================

import os
import json
import time
import logging
import warnings
from datetime import datetime, timedelta

import requests
import numpy as np
import pandas as pd
import xgboost as xgb
import joblib
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score

warnings.filterwarnings("ignore")
np.random.seed(42)

# ══════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════

# 🔐 لا تكتب المفتاح هنا. اضبطه كمتغير بيئة قبل التشغيل:
#    export TWELVE_DATA_API_KEY="your_key_here"     (Linux/Mac)
#    os.environ["TWELVE_DATA_API_KEY"] = "..."      (Colab cell منفصل، لا تشاركه)
TWELVE_DATA_API_KEY = os.environ.get("TWELVE_DATA_API_KEY","")
if not TWELVE_DATA_API_KEY:
    raise RuntimeError(
        "TWELVE_DATA_API_KEY غير موجود. اضبطه كمتغير بيئة قبل التشغيل ولا "
        "تكتبه أبداً داخل الكود."
    )

SYMBOL      = "USD/JPY"      # الفضة بدل الدهب
SYMBOL_SAFE = "USDJPY"
# مرجع الدهب — بيتسحب بس عشان نحسب منه فيتشور "نسبة الدهب/الفضة"
# (Gold/Silver Ratio)، مؤشر كلاسيكي في تداول المعادن النفيسة.
# REFERENCE_METAL_SYMBOL not needed for USD/JPY forex pair
MODELS_DIR  = "models/ict_usdjpy"
DATA_DIR    = "data/ict_usdjpy"
HOLDOUT_PCT = 0.15   # آخر 15% من كل فريم — معزولة تماماً عن التدريب

# ─────────────────────────────────────────────────────────────
# TIMEFRAME CONFIG
# ─────────────────────────────────────────────────────────────
# ─── باريمترات USD/JPY المُحسَّنة لمدرسة ICT ───
# USD/JPY خصائص فريدة تفصله عن EUR/USD و AUD/USD:
#   - **مقياس السعر ~100x** (~150 مقابل ~1.10) — كل تفاوتات النسبة المطلقة
#     (زي eq_tol) لازم تتضرب في ~100 عشان تعبّر عن نفس عدد النقط الحقيقي
#   - تاني أكثر زوج سيولة في العالم (سبريد ضيق ~1 bps)
#   - **جلسة طوكيو نشيطة فعلياً** (BoJ + مصدّرين يابانيين + Tokyo Fix) —
#     عكس EUR/USD اللي طوكيو فيه هادية. ده معناه فرص تداول أكتر خلال 24 ساعة
#   - **BoJ (سياسة YCC وتدخلات وزارة المالية) + عائد سندات الخزانة 10Y
#     الأمريكية** هما المحرك الماكرو الأساسي — مش بيانات المنطقة اليورو
#   - **حساسية لموجات ملاذ آمن**: JPY بيقوى فجأة في الأزمات → سلوك غير خطي
#   - محتاجين ATR SL/TP أوسع شوية من EUR/USD للحماية من قفزات BoJ المفاجئة
TIMEFRAME_CONFIG = {
    "5min": {
        "interval": "5min", "years": 3,
        "buy_threshold": 0.58, "sell_threshold": 0.38,
        "fold_buy_threshold": 0.57, "fold_sell_threshold": 0.43,
        "max_trades_per_day": 12,   # طوكيو + لندن-NY كلاهما نشيط
        "larger_tf": "1h",
        "horizon": 10, "swing_lookback": 5, "target_atr_mult": 0.5,
        "min_candles": 1500, "confidence_threshold": 0.60,
        "atr_sl_mult": 1.1, "atr_tp_mult": 1.9,   # وسّعنا شوية لخطر BoJ
    },
    "15min": {
        "interval": "15min", "years": 4,
        "buy_threshold": 0.58, "sell_threshold": 0.37,
        "fold_buy_threshold": 0.56, "fold_sell_threshold": 0.42,
        "max_trades_per_day": 8,
        "larger_tf": "4h",
        "horizon": 8, "swing_lookback": 5, "target_atr_mult": 0.5,
        "min_candles": 1000, "confidence_threshold": 0.60,
        "atr_sl_mult": 1.2, "atr_tp_mult": 2.0,
    },
    "45min": {
        "interval": "45min", "years": 5,
        "buy_threshold": 0.58, "sell_threshold": 0.39,
        "fold_buy_threshold": 0.56, "fold_sell_threshold": 0.44,
        "max_trades_per_day": 6,
        "larger_tf": "4h",
        "horizon": 6, "swing_lookback": 4, "target_atr_mult": 0.45,
        "min_candles": 700, "confidence_threshold": 0.62,
        "atr_sl_mult": 1.2, "atr_tp_mult": 2.1,
    },
    "1h": {
        "interval": "1h", "years": 7,
        "buy_threshold": 0.58, "sell_threshold": 0.42,
        "fold_buy_threshold": 0.54, "fold_sell_threshold": 0.46,
        "max_trades_per_day": 5,
        "larger_tf": "4h",
        "horizon": 14, "swing_lookback": 4, "target_atr_mult": 0.4,
        "min_candles": 700, "confidence_threshold": 0.63,
        "atr_sl_mult": 1.15, "atr_tp_mult": 1.8,
    },
    "2h": {
        "interval": "2h", "years": 8,
        "buy_threshold": 0.56, "sell_threshold": 0.44,
        "fold_buy_threshold": 0.53, "fold_sell_threshold": 0.47,
        "max_trades_per_day": 3,
        "larger_tf": "1d",
        "horizon": 14, "swing_lookback": 4, "target_atr_mult": 0.35,
        "min_candles": 500, "confidence_threshold": 0.65,
        "atr_sl_mult": 1.15, "atr_tp_mult": 1.8, "calib_percentile": 80,
        "cv_splits": 4,
    },
    "4h": {
        "interval": "4h", "years": 7,
        "buy_threshold": 0.56, "sell_threshold": 0.43,
        "fold_buy_threshold": 0.54, "fold_sell_threshold": 0.46,
        "max_trades_per_day": 2,
        "larger_tf": "1d",
        "horizon": 14, "swing_lookback": 3, "target_atr_mult": 0.3,
        "min_candles": 400, "confidence_threshold": 0.66,
        "atr_sl_mult": 1.2, "atr_tp_mult": 1.85, "calib_percentile": 80,
        "cv_splits": 4, "force_resample_source": "1h",
    },
    "1d": {
        "interval": "1day", "years": 12,
        "buy_threshold": 0.54, "sell_threshold": 0.45,
        "fold_buy_threshold": 0.53, "fold_sell_threshold": 0.47,
        "max_trades_per_day": 1,
        "larger_tf": "1W",
        "horizon": 10, "swing_lookback": 3, "target_atr_mult": 0.4,
        "min_candles": 300, "confidence_threshold": 0.57,
        "atr_sl_mult": 1.2, "atr_tp_mult": 1.7, "calib_percentile": 82,
        "min_reliable_trades": 15, "cv_splits": 3,
    },
    "1W": {
        "interval": "1week", "years": 18,
        "buy_threshold": 0.53, "sell_threshold": 0.46,
        "fold_buy_threshold": 0.52, "fold_sell_threshold": 0.48,
        "max_trades_per_day": 1,
        "larger_tf": None,
        "horizon": 8, "swing_lookback": 2, "target_atr_mult": 0.35,
        "min_candles": 100, "confidence_threshold": 0.54,
        "atr_sl_mult": 1.3, "atr_tp_mult": 1.8, "calib_percentile": 78,
        "min_reliable_trades": 10, "cv_splits": 3,
    },
}
# قيمة افتراضية للفريمات اللي معندهاش min_reliable_trades محدد صراحة
for _tf_cfg in TIMEFRAME_CONFIG.values():
    _tf_cfg.setdefault("min_reliable_trades", 15)
    _tf_cfg.setdefault("calib_percentile", 88)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.FileHandler("training_ict_usdjpy.log"), logging.StreamHandler()],
)

# فحص فوري: كل larger_tf لازم يكون مفتاح موجود فعلاً في TIMEFRAME_CONFIG
for _tf, _cfg in TIMEFRAME_CONFIG.items():
    _ltf = _cfg.get("larger_tf")
    if _ltf is not None and _ltf not in TIMEFRAME_CONFIG:
        raise ValueError(
            f"TIMEFRAME_CONFIG['{_tf}']['larger_tf'] = '{_ltf}' غير موجود كمفتاح "
            f"في TIMEFRAME_CONFIG. المفاتيح المتاحة: {list(TIMEFRAME_CONFIG.keys())}"
        )

# ══════════════════════════════════════════════════════════════
# DATA FETCHER
# ══════════════════════════════════════════════════════════════
class TwelveDataFetcher:
    BASE_URL = "https://api.twelvedata.com/time_series"

    def __init__(self, api_key):
        self.api_key = api_key

    def fetch_range(self, symbol, interval, start_date=None, end_date=None, outputsize=5000):
        params = {
            "symbol": symbol, "interval": interval,
            "apikey": self.api_key, "format": "JSON",
            "order": "ASC", "outputsize": outputsize,
        }
        if start_date:
            params["start_date"] = start_date
        if end_date:
            params["end_date"] = end_date
        try:
            r = requests.get(self.BASE_URL, params=params, timeout=30)
            if r.status_code != 200:
                try:
                    body = r.json()
                except Exception:
                    body = {}
                body_msg = body.get("message", "")
                if r.status_code == 400 and "No data is available" in body_msg:
                    logging.info(f"[{interval}] وصلنا لأقدم بيانات متاحة على الخطة الحالية.")
                elif r.status_code == 404:
                    logging.info(
                        f"[{interval}] 404 من الـ API — الفريم/التركيبة دي على الأغلب "
                        f"مش مدعومة مباشرة على الخطة الحالية، هنعتمد على resample fallback."
                    )
                else:
                    logging.error(f"خطأ اتصال {r.status_code}: {r.text[:200]}")
                return pd.DataFrame()
            data = r.json()
            if "values" not in data:
                msg = data.get("message", str(data))
                if data.get("code") == 400 and "No data is available" in msg:
                    logging.info(f"[{interval}] وصلنا لأقدم بيانات متاحة على الخطة الحالية.")
                else:
                    logging.error(f"لا بيانات ({interval}): {msg}")
                return pd.DataFrame()
            df = pd.DataFrame(data["values"])
            if "volume" not in df.columns:
                df["volume"] = 0
            for col in ["open", "high", "low", "close", "volume"]:
                df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
            df["datetime"] = pd.to_datetime(df["datetime"])
            df = df.sort_values("datetime").reset_index(drop=True)
            return df
        except Exception as e:
            logging.error(f"استثناء أثناء الجلب: {e}")
            return pd.DataFrame()


def fetch_native_history(interval, years, symbol=SYMBOL, max_requests=40):
    fetcher = TwelveDataFetcher(TWELVE_DATA_API_KEY)
    cutoff = datetime.utcnow() - timedelta(days=int(365 * years))
    end_date = datetime.utcnow()
    all_data, previous_oldest, req = [], None, 0

    while req < max_requests:
        req += 1
        df_batch = fetcher.fetch_range(
            symbol=symbol, interval=interval,
            end_date=end_date.strftime("%Y-%m-%d %H:%M:%S"),
            outputsize=5000,
        )
        if df_batch.empty:
            break
        all_data.append(df_batch)
        oldest = df_batch["datetime"].min()
        logging.info(f"[{interval}] جلب دفعة حتى {oldest} — إجمالي دفعات: {len(all_data)}")
        if previous_oldest == oldest or oldest <= cutoff:
            break
        previous_oldest = oldest
        end_date = oldest - timedelta(seconds=1)
        time.sleep(0.6)

    if not all_data:
        return pd.DataFrame()
    df = pd.concat(all_data, ignore_index=True)
    df.drop_duplicates(subset="datetime", keep="first", inplace=True)
    df = df[df["datetime"] >= cutoff].sort_values("datetime").reset_index(drop=True)

    if not df.empty:
        actual_years = (df["datetime"].max() - df["datetime"].min()).days / 365.0
        logging.info(f"[{interval}] إجمالي الشموع بعد القطع لـ {years} سنة مطلوبة: {len(df)} "
                      f"(تغطية فعلية: {actual_years:.2f} سنة)")
        if actual_years < years * 0.7:
            logging.warning(
                f"[{interval}] ⚠️ التغطية الفعلية ({actual_years:.2f} سنة) أقل بكتير من "
                f"المطلوب ({years} سنة) — الخطة الحالية في TwelveData على الأرجح محدودة "
                f"لهذا الـ interval. إما قلّل قيمة 'years' في TIMEFRAME_CONFIG لتطابق الواقع، "
                f"أو رقّي خطة TwelveData. تدريب على تاريخ قصير بيضعّف جودة الإشارة ويقلل "
                f"عدد الصفقات في كل فولد."
            )
    return df


# ─── USD/JPY: نسبة الدهب/الفضة غير منطقية هنا (زوج فوركس، مش معدن) ───
# الدالتين دول اتعطلوا بالكامل (no-op) بدل ما يتشالوا، عشان باقي الكود
# (اللي بينادي عليهم في get_or_fetch_timeframe_data) يفضل شغال من غير
# تعديل كل نقطة استدعاء على حدة.
_REFERENCE_METAL_CACHE = {}


def fetch_reference_metal_history(interval, years):
    return pd.DataFrame()


def attach_gold_reference(df, config):
    return df


def get_or_fetch_timeframe_data(tf_key, config, force_refresh=False):
    os.makedirs(DATA_DIR, exist_ok=True)
    csv_path = os.path.join(DATA_DIR, f"{SYMBOL_SAFE}_{tf_key}_raw.csv")
    if os.path.exists(csv_path) and not force_refresh:
        df_old = pd.read_csv(csv_path)
        df_old["datetime"] = pd.to_datetime(df_old["datetime"])
        fetcher = TwelveDataFetcher(TWELVE_DATA_API_KEY)
        last_date = df_old["datetime"].max()
        df_new = fetcher.fetch_range(
            symbol=SYMBOL, interval=config["interval"],
            start_date=(last_date + timedelta(minutes=1)).strftime("%Y-%m-%d %H:%M:%S"),
            end_date=datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S"),
            outputsize=2000,
        )
        if not df_new.empty:
            df_new = attach_gold_reference(df_new, config)
            df_old = pd.concat([df_old, df_new], ignore_index=True)
            df_old.drop_duplicates(subset="datetime", keep="last", inplace=True)
            df_old.sort_values("datetime", inplace=True)
            df_old.reset_index(drop=True, inplace=True)
            df_old.to_csv(csv_path, index=False)
        return df_old

    df = fetch_native_history(config["interval"], config["years"])

    if df.empty:
        pandas_freq_map = {
            "5min": "5min", "15min": "15min", "45min": "45min",
            "1h": "1h", "2h": "2h", "4h": "4h", "1day": "1D", "1week": "1W",
        }
        rule = pandas_freq_map.get(config["interval"])
        fallback_chain = {
            "2h": ["1h", "45min", "15min", "5min"],
            "4h": ["1h", "45min", "15min", "5min"],
            "1day": ["4h", "1h", "45min", "15min"],
            "1week": ["1d", "4h", "1h"],
        }
        smaller_candidates = fallback_chain.get(config["interval"], ["15min", "5min"])
        for smaller_tf in smaller_candidates:
            if smaller_tf == tf_key:
                continue
            smaller_csv = os.path.join(DATA_DIR, f"{SYMBOL_SAFE}_{smaller_tf}_raw.csv")
            if os.path.exists(smaller_csv):
                df_small = pd.read_csv(smaller_csv)
                df_small["datetime"] = pd.to_datetime(df_small["datetime"])
            elif smaller_tf in TIMEFRAME_CONFIG:
                df_small = fetch_native_history(
                    TIMEFRAME_CONFIG[smaller_tf]["interval"],
                    TIMEFRAME_CONFIG[smaller_tf]["years"],
                )
            else:
                continue
            if not df_small.empty and rule:
                logging.warning(
                    f"[{tf_key}] الجلب المباشر رجع فاضي (حد تاريخي حقيقي في الخطة الحالية) "
                    f"— هعمل resample من {smaller_tf} كحل بديل. التغطية التاريخية هتكون "
                    f"بحدود {smaller_tf} نفسه، مش أفضل."
                )
                df = (
                    df_small.set_index("datetime")
                    .resample(rule)
                    .agg({"open": "first", "high": "max", "low": "min",
                          "close": "last", "volume": "sum"})
                    .dropna()
                    .reset_index()
                )
                break

    if not df.empty:
        df = attach_gold_reference(df, config)
        df.to_csv(csv_path, index=False)
    else:
        logging.error(f"[{tf_key}] فشل الحصول على أي بيانات (لا مباشر ولا resample fallback)")
    return df


# ══════════════════════════════════════════════════════════════
# ICT FEATURE ENGINEER
# ══════════════════════════════════════════════════════════════
class ICTFeatureEngineer:
    def __init__(self):
        self.scaler = RobustScaler()
        self.feature_columns = None
        self.scaler_fitted = False

    def add_sessions_and_killzones(self, df):
        df = df.copy()
        dt = df["datetime"]
        h = dt.dt.hour
        df["hour"] = h
        df["day_of_week"] = dt.dt.dayofweek
        df["hour_sin"] = np.sin(2 * np.pi * h / 24)
        df["hour_cos"] = np.cos(2 * np.pi * h / 24)

        # USD/JPY sessions — طوكيو ولندن-NY overlap كلاهما نشيط
        df["is_tokyo"]  = ((h >= 0) & (h < 6)).astype(int)   # جلسة طوكيو
        df["is_asia"]   = ((h >= 0) & (h < 7)).astype(int)
        df["is_london"] = ((h >= 7) & (h < 16)).astype(int)
        df["is_ny"]     = ((h >= 12) & (h < 21)).astype(int)
        df["is_overlap"]= ((h >= 12) & (h < 16)).astype(int)
        df["is_boj_window"] = ((h >= 2) & (h < 4)).astype(int)
        df["is_us_data_window"] = ((h >= 12) & (h < 14)).astype(int)
        df["is_tokyo_fix"] = ((h == 0) & (dt.dt.minute < 30)).astype(int)
        df["is_london_ny_prime"] = ((h >= 12) & (h < 16)).astype(int)

        df["kz_london_open"] = ((h >= 7) & (h < 10)).astype(int)
        df["kz_ny_am"]       = ((h >= 12) & (h < 15)).astype(int)
        df["kz_london_close"]= ((h >= 14) & (h < 16)).astype(int)
        df["kz_silver_bullet_am"] = ((h == 14)).astype(int)
        df["kz_silver_bullet_pm"] = ((h == 19)).astype(int)
        df["is_friday"] = (df["day_of_week"] == 4).astype(int)

        df["date_only"] = dt.dt.date
        asia_mask = df["is_asia"] == 1
        asia_high = df.loc[asia_mask].groupby("date_only")["high"].transform("max")
        asia_low  = df.loc[asia_mask].groupby("date_only")["low"].transform("min")
        df["asia_high"] = np.nan
        df["asia_low"] = np.nan
        df.loc[asia_mask, "asia_high"] = asia_high
        df.loc[asia_mask, "asia_low"] = asia_low
        df["asia_high"] = df.groupby("date_only")["asia_high"].transform(
            lambda s: s.ffill().bfill()
        )
        df["asia_low"] = df.groupby("date_only")["asia_low"].transform(
            lambda s: s.ffill().bfill()
        )
        df["dist_to_asia_high"] = (df["asia_high"] - df["close"]) / (df["close"] + 1e-9)
        df["dist_to_asia_low"]  = (df["close"] - df["asia_low"]) / (df["close"] + 1e-9)
        df.drop(columns=["date_only"], inplace=True)
        return df

    def compute_base_indicators(self, df):
        df = df.copy()
        c, h, l, o = df["close"], df["high"], df["low"], df["open"]

        tr = pd.concat([
            h - l, (h - c.shift()).abs(), (l - c.shift()).abs()
        ], axis=1).max(axis=1)
        df["atr"] = tr.rolling(14).mean()
        df["atr_pct"] = df["atr"] / c

        df["ema_50"] = c.ewm(span=50).mean()
        df["ema_200"] = c.ewm(span=200).mean()
        df["trend_filter"] = (df["ema_50"] > df["ema_200"]).astype(int)

        df["body_size"] = (c - o).abs()
        df["candle_range"] = (h - l)
        df["upper_wick"] = h - c.where(c >= o, o)
        df["lower_wick"] = c.where(c <= o, o) - l
        df["close_position"] = (c - l) / (h - l + 1e-9)
        df["returns"] = c.pct_change()
        return df

    def add_gold_silver_ratio(self, df):
        df = df.copy()
        if "xau_close" not in df.columns:
            return df
        silver_close_safe = df["close"].replace(0, np.nan)
        ratio = df["xau_close"] / silver_close_safe
        ratio_ma = ratio.rolling(50, min_periods=10).mean()
        ratio_std = ratio.rolling(50, min_periods=10).std()

        df["gold_silver_ratio"] = ratio
        df["gold_silver_ratio_ma50"] = ratio_ma
        df["gold_silver_ratio_z"] = ((ratio - ratio_ma) / (ratio_std + 1e-9)).clip(-4, 4)
        df["gold_silver_ratio_chg_10"] = ratio.pct_change(10)

        if "returns" in df.columns:
            xau_returns = df["xau_close"].pct_change()
            df["gold_silver_return_diff"] = df["returns"] - xau_returns

        df.drop(columns=["xau_close"], inplace=True)
        return df

    def add_swing_points(self, df, lookback=5):
        df = df.copy()
        h, l = df["high"], df["low"]
        n = len(df)
        is_sw_high = np.zeros(n, dtype=int)
        is_sw_low  = np.zeros(n, dtype=int)

        for i in range(lookback, n - lookback):
            window_h = h.iloc[i - lookback:i + lookback + 1]
            window_l = l.iloc[i - lookback:i + lookback + 1]
            if h.iloc[i] == window_h.max() and (window_h == h.iloc[i]).sum() == 1:
                is_sw_high[i] = 1
            if l.iloc[i] == window_l.min() and (window_l == l.iloc[i]).sum() == 1:
                is_sw_low[i] = 1

        confirmed_high = pd.Series(is_sw_high, index=df.index).shift(lookback).fillna(0).astype(int)
        confirmed_low  = pd.Series(is_sw_low,  index=df.index).shift(lookback).fillna(0).astype(int)

        df["swing_high_flag"] = confirmed_high
        df["swing_low_flag"] = confirmed_low
        df["last_swing_high"] = df["high"].where(confirmed_high == 1).ffill()
        df["last_swing_low"]  = df["low"].where(confirmed_low == 1).ffill()
        return df

    def add_market_structure_ict(self, df):
        df = df.copy()
        c = df["close"]
        prev_sw_high = df["last_swing_high"].shift(1)
        prev_sw_low  = df["last_swing_low"].shift(1)

        df["bos_up"]   = (c > prev_sw_high).astype(int)
        df["bos_down"] = (c < prev_sw_low).astype(int)

        structure = np.zeros(len(df), dtype=int)
        cur = 0
        bos_up = df["bos_up"].values
        bos_down = df["bos_down"].values
        for i in range(len(df)):
            if bos_up[i] == 1:
                cur = 1
            elif bos_down[i] == 1:
                cur = -1
            structure[i] = cur
        df["market_structure_trend"] = structure

        prev_trend = pd.Series(structure, index=df.index).shift(1).fillna(0)
        df["choch_bullish"] = ((df["bos_up"] == 1) & (prev_trend == -1)).astype(int)
        df["choch_bearish"] = ((df["bos_down"] == 1) & (prev_trend == 1)).astype(int)

        df["dist_to_last_swing_high"] = (df["last_swing_high"] - c) / (c + 1e-9)
        df["dist_to_last_swing_low"]  = (c - df["last_swing_low"]) / (c + 1e-9)
        return df

    def add_fair_value_gaps(self, df):
        df = df.copy()
        h, l, c = df["high"].values, df["low"].values, df["close"].values
        n = len(df)
        fvg_bull = np.zeros(n); fvg_bear = np.zeros(n)
        fvg_bull_top = np.full(n, np.nan); fvg_bull_bot = np.full(n, np.nan)
        fvg_bear_top = np.full(n, np.nan); fvg_bear_bot = np.full(n, np.nan)

        for i in range(2, n):
            if l[i] > h[i - 2]:
                fvg_bull[i] = 1
                fvg_bull_top[i] = l[i]
                fvg_bull_bot[i] = h[i - 2]
            if h[i] < l[i - 2]:
                fvg_bear[i] = 1
                fvg_bear_top[i] = l[i - 2]
                fvg_bear_bot[i] = h[i]

        df["fvg_bullish"] = fvg_bull.astype(int)
        df["fvg_bearish"] = fvg_bear.astype(int)

        last_bull_top = pd.Series(fvg_bull_top, index=df.index).ffill()
        last_bull_bot = pd.Series(fvg_bull_bot, index=df.index).ffill()
        last_bear_top = pd.Series(fvg_bear_top, index=df.index).ffill()
        last_bear_bot = pd.Series(fvg_bear_bot, index=df.index).ffill()

        c_s = df["close"]
        bull_range = (last_bull_top - last_bull_bot).replace(0, np.nan)
        bear_range = (last_bear_top - last_bear_bot).replace(0, np.nan)
        df["dist_to_bull_fvg"] = (c_s - last_bull_bot) / (c_s + 1e-9)
        df["dist_to_bear_fvg"] = (last_bear_top - c_s) / (c_s + 1e-9)
        df["bull_fvg_fill_pct"] = ((last_bull_top - c_s) / bull_range).clip(0, 1).fillna(0)
        df["bear_fvg_fill_pct"] = ((c_s - last_bear_bot) / bear_range).clip(0, 1).fillna(0)
        return df

    def add_order_blocks(self, df, disp_mult=1.5, max_lookback=6):
        df = df.copy()
        o, c, h, l = df["open"].values, df["close"].values, df["high"].values, df["low"].values
        atr = df["atr"].values
        bos_up = df["bos_up"].values
        bos_down = df["bos_down"].values
        n = len(df)

        bull_ob_top = np.full(n, np.nan); bull_ob_bot = np.full(n, np.nan)
        bear_ob_top = np.full(n, np.nan); bear_ob_bot = np.full(n, np.nan)
        bull_ob_strength = np.full(n, np.nan); bear_ob_strength = np.full(n, np.nan)
        bull_mitigated = np.zeros(n, dtype=int); bear_mitigated = np.zeros(n, dtype=int)
        bull_breaker = np.zeros(n, dtype=int); bear_breaker = np.zeros(n, dtype=int)

        cur_bull_top = cur_bull_bot = np.nan
        cur_bear_top = cur_bear_bot = np.nan
        cur_bull_mitigated = cur_bear_mitigated = False
        cur_bull_broken = cur_bear_broken = False

        for i in range(1, n):
            body = abs(c[i] - o[i])
            is_displacement = (not np.isnan(atr[i])) and atr[i] > 0 and body > disp_mult * atr[i]

            if bos_up[i] == 1 and is_displacement and c[i] > o[i]:
                for back in range(1, max_lookback + 1):
                    j = i - back
                    if j < 0:
                        break
                    if c[j] < o[j]:
                        cur_bull_top, cur_bull_bot = h[j], l[j]
                        cur_bull_mitigated, cur_bull_broken = False, False
                        break
                    if c[j] > o[j] and back > 1:
                        break

            if bos_down[i] == 1 and is_displacement and c[i] < o[i]:
                for back in range(1, max_lookback + 1):
                    j = i - back
                    if j < 0:
                        break
                    if c[j] > o[j]:
                        cur_bear_top, cur_bear_bot = h[j], l[j]
                        cur_bear_mitigated, cur_bear_broken = False, False
                        break
                    if c[j] < o[j] and back > 1:
                        break

            if not np.isnan(cur_bull_bot) and l[i] <= cur_bull_top and not cur_bull_mitigated:
                cur_bull_mitigated = True
            if not np.isnan(cur_bear_top) and h[i] >= cur_bear_bot and not cur_bear_mitigated:
                cur_bear_mitigated = True

            if cur_bull_mitigated and not np.isnan(cur_bull_bot) and c[i] < cur_bull_bot:
                cur_bull_broken = True
            if cur_bear_mitigated and not np.isnan(cur_bear_top) and c[i] > cur_bear_top:
                cur_bear_broken = True

            bull_ob_top[i], bull_ob_bot[i] = cur_bull_top, cur_bull_bot
            bear_ob_top[i], bear_ob_bot[i] = cur_bear_top, cur_bear_bot
            if not np.isnan(atr[i]) and atr[i] > 0:
                if not np.isnan(cur_bull_top):
                    bull_ob_strength[i] = abs(c[i - 1] - o[i - 1]) / atr[i] if i > 0 else np.nan
                if not np.isnan(cur_bear_top):
                    bear_ob_strength[i] = abs(c[i - 1] - o[i - 1]) / atr[i] if i > 0 else np.nan
            bull_mitigated[i] = int(cur_bull_mitigated)
            bear_mitigated[i] = int(cur_bear_mitigated)
            bull_breaker[i] = int(cur_bull_broken)
            bear_breaker[i] = int(cur_bear_broken)

        c_s = df["close"]
        df["bull_ob_top"] = bull_ob_top
        df["bull_ob_bot"] = bull_ob_bot
        df["bear_ob_top"] = bear_ob_top
        df["bear_ob_bot"] = bear_ob_bot
        df["bull_ob_strength"] = bull_ob_strength
        df["bear_ob_strength"] = bear_ob_strength
        df["bull_ob_mitigated"] = bull_mitigated
        df["bear_ob_mitigated"] = bear_mitigated
        df["bull_breaker_block"] = bull_breaker
        df["bear_breaker_block"] = bear_breaker
        df["dist_to_bull_ob"] = (c_s - df["bull_ob_bot"]) / (c_s + 1e-9)
        df["dist_to_bear_ob"] = (df["bear_ob_top"] - c_s) / (c_s + 1e-9)
        df["price_inside_bull_ob"] = (
            (c_s <= df["bull_ob_top"]) & (c_s >= df["bull_ob_bot"]) & (bull_mitigated == 0)
        ).astype(int)
        df["price_inside_bear_ob"] = (
            (c_s <= df["bear_ob_top"]) & (c_s >= df["bear_ob_bot"]) & (bear_mitigated == 0)
        ).astype(int)

        df["in_mitigation_zone_bull"] = (
            (c_s <= df["bull_ob_top"]) & (c_s >= df["bull_ob_bot"]) & (bull_mitigated == 1)
        ).astype(int)
        df["in_mitigation_zone_bear"] = (
            (c_s <= df["bear_ob_top"]) & (c_s >= df["bear_ob_bot"]) & (bear_mitigated == 1)
        ).astype(int)

        bull_breaker_level = pd.Series(
            np.where(bull_breaker == 1, bull_ob_bot, np.nan), index=df.index
        ).ffill()
        bear_breaker_level = pd.Series(
            np.where(bear_breaker == 1, bear_ob_top, np.nan), index=df.index
        ).ffill()
        df["dist_to_bull_breaker"] = ((c_s - bull_breaker_level) / (c_s + 1e-9)).fillna(0)
        df["dist_to_bear_breaker"] = ((bear_breaker_level - c_s) / (c_s + 1e-9)).fillna(0)
        return df

    def add_liquidity(self, df, eq_tol=0.04, sweep_lookback=20):
        df = df.copy()
        h, l, c = df["high"], df["low"], df["close"]

        roll_high = h.rolling(sweep_lookback).max()
        roll_low = l.rolling(sweep_lookback).min()

        df["eq_highs"] = ((roll_high - h).abs() / (c + 1e-9) < eq_tol).astype(int)
        df["eq_lows"]  = ((roll_low - l).abs() / (c + 1e-9) < eq_tol).astype(int)

        prev_roll_high = roll_high.shift(1)
        prev_roll_low = roll_low.shift(1)
        df["liquidity_sweep_up"] = (
            (h > prev_roll_high) & (c < prev_roll_high)
        ).astype(int)
        df["liquidity_sweep_down"] = (
            (l < prev_roll_low) & (c > prev_roll_low)
        ).astype(int)

        df["dist_to_buyside_liquidity"] = (roll_high - c) / (c + 1e-9)
        df["dist_to_sellside_liquidity"] = (c - roll_low) / (c + 1e-9)
        return df

    def add_premium_discount(self, df):
        df = df.copy()
        c = df["close"]
        swing_high = df["last_swing_high"]
        swing_low = df["last_swing_low"]
        rng = (swing_high - swing_low).replace(0, np.nan)

        pos = (c - swing_low) / rng
        df["pd_position"] = pos.clip(0, 1).fillna(0.5)
        df["is_premium"] = (df["pd_position"] > 0.5).astype(int)
        df["is_discount"] = (df["pd_position"] <= 0.5).astype(int)
        df["in_ote_bullish"] = ((df["pd_position"] >= 0.21) & (df["pd_position"] <= 0.38)).astype(int)
        df["in_ote_bearish"] = ((df["pd_position"] >= 0.62) & (df["pd_position"] <= 0.79)).astype(int)
        return df

    def add_displacement(self, df, mult=1.5):
        df = df.copy()
        body = df["body_size"]
        df["displacement"] = (body > mult * df["atr"]).astype(int)
        df["displacement_bullish"] = (
            (df["displacement"] == 1) & (df["close"] > df["open"])
        ).astype(int)
        df["displacement_bearish"] = (
            (df["displacement"] == 1) & (df["close"] < df["open"])
        ).astype(int)

        next_close = df["close"].shift(-1)
        next_open = df["open"].shift(-1)
        df["displacement_confirmed_bull"] = (
            (df["displacement_bullish"] == 1) & (next_close > next_open) &
            (next_close > df["close"])
        ).astype(int)
        df["displacement_confirmed_bear"] = (
            (df["displacement_bearish"] == 1) & (next_close < next_open) &
            (next_close < df["close"])
        ).astype(int)
        return df

    def add_inducement(self, df, minor_lookback=2, revert_window=3):
        df = df.copy()
        h, l, c = df["high"].values, df["low"].values, df["close"].values
        n = len(df)
        is_minor_high = np.zeros(n, dtype=int)
        is_minor_low = np.zeros(n, dtype=int)
        for i in range(minor_lookback, n - minor_lookback):
            wh = h[i - minor_lookback:i + minor_lookback + 1]
            wl = l[i - minor_lookback:i + minor_lookback + 1]
            if h[i] == wh.max() and (wh == h[i]).sum() == 1:
                is_minor_high[i] = 1
            if l[i] == wl.min() and (wl == l[i]).sum() == 1:
                is_minor_low[i] = 1
        confirmed_minor_high = pd.Series(is_minor_high, index=df.index).shift(minor_lookback).fillna(0).astype(int)
        confirmed_minor_low = pd.Series(is_minor_low, index=df.index).shift(minor_lookback).fillna(0).astype(int)
        last_minor_high = df["high"].where(confirmed_minor_high == 1).ffill()
        last_minor_low = df["low"].where(confirmed_minor_low == 1).ffill()

        inducement_bull = np.zeros(n, dtype=int)
        inducement_bear = np.zeros(n, dtype=int)
        lml = last_minor_low.values
        lmh = last_minor_high.values
        for i in range(revert_window, n):
            if not np.isnan(lml[i - revert_window]) and l[i - revert_window:i].min() < lml[i - revert_window] and c[i] > lml[i - revert_window]:
                inducement_bull[i] = 1
            if not np.isnan(lmh[i - revert_window]) and h[i - revert_window:i].max() > lmh[i - revert_window] and c[i] < lmh[i - revert_window]:
                inducement_bear[i] = 1

        df["inducement_bull"] = inducement_bull
        df["inducement_bear"] = inducement_bear
        return df

    def add_liquidity_void_raid(self, df, void_atr_mult=2.0, raid_atr_mult=0.5):
        df = df.copy()
        atr_safe = df["atr"].replace(0, np.nan)
        df["liquidity_void"] = (df["candle_range"] > void_atr_mult * df["atr"]).astype(int)
        void_top = df["high"].where(df["liquidity_void"] == 1).ffill()
        void_bot = df["low"].where(df["liquidity_void"] == 1).ffill()
        df["dist_to_liquidity_void"] = ((df["close"] - void_bot) / (df["close"] + 1e-9)).fillna(0)

        upper_wick_atr = (df.get("upper_wick", pd.Series(0, index=df.index)) / atr_safe).fillna(0)
        lower_wick_atr = (df.get("lower_wick", pd.Series(0, index=df.index)) / atr_safe).fillna(0)
        df["liquidity_raid_up"] = (
            (df.get("liquidity_sweep_up", 0) == 1) & (upper_wick_atr > raid_atr_mult)
        ).astype(int)
        df["liquidity_raid_down"] = (
            (df.get("liquidity_sweep_down", 0) == 1) & (lower_wick_atr > raid_atr_mult)
        ).astype(int)
        return df

    def add_session_momentum(self, df):
        df = df.copy()
        df["_date_only"] = df["datetime"].dt.date
        df["_session_id"] = (
            df["_date_only"].astype(str) + "_" +
            np.where(df["is_asia"] == 1, "asia",
            np.where(df["is_london"] == 1, "london",
            np.where(df["is_ny"] == 1, "ny", "other")))
        )
        session_open = df.groupby("_session_id")["open"].transform("first")
        df["session_momentum"] = ((df["close"] - session_open) / (session_open + 1e-9))
        df["session_momentum_atr"] = (df["close"] - session_open) / (df["atr"].replace(0, np.nan) + 1e-9)
        df["session_momentum_atr"] = df["session_momentum_atr"].fillna(0)
        df.drop(columns=["_date_only", "_session_id"], inplace=True)
        return df

    def add_confluence_score(self, df):
        df = df.copy()
        fresh_bull_ob = (df.get("price_inside_bull_ob", 0) == 1).astype(int)
        fresh_bear_ob = (df.get("price_inside_bear_ob", 0) == 1).astype(int)
        ob_strength_bull = df.get("bull_ob_strength", pd.Series(0, index=df.index)).fillna(0).clip(0, 5) / 5
        ob_strength_bear = df.get("bear_ob_strength", pd.Series(0, index=df.index)).fillna(0).clip(0, 5) / 5

        fvg_bull_freshness = (1 - df.get("bull_fvg_fill_pct", pd.Series(0, index=df.index)).fillna(0)).clip(0, 1)
        fvg_bear_freshness = (1 - df.get("bear_fvg_fill_pct", pd.Series(0, index=df.index)).fillna(0)).clip(0, 1)
        fvg_bull_component = df.get("fvg_bullish", 0) * 1.0 + (1 - df.get("fvg_bullish", 0)) * (
            0.4 * fvg_bull_freshness * (df.get("dist_to_bull_fvg", pd.Series(0, index=df.index)).abs() < 0.003).astype(int)
        )
        fvg_bear_component = df.get("fvg_bearish", 0) * 1.0 + (1 - df.get("fvg_bearish", 0)) * (
            0.4 * fvg_bear_freshness * (df.get("dist_to_bear_fvg", pd.Series(0, index=df.index)).abs() < 0.003).astype(int)
        )

        pd_pos = df.get("pd_position", pd.Series(0.5, index=df.index))
        pd_align_bull = (1 - pd_pos).clip(0, 1)
        pd_align_bear = pd_pos.clip(0, 1)

        atr_safe = df.get("atr", pd.Series(np.nan, index=df.index)).replace(0, np.nan)
        upper_wick_atr = (df.get("upper_wick", pd.Series(0, index=df.index)) / atr_safe).fillna(0).clip(0, 3) / 3
        lower_wick_atr = (df.get("lower_wick", pd.Series(0, index=df.index)) / atr_safe).fillna(0).clip(0, 3) / 3
        sweep_down_strength = df.get("liquidity_sweep_down", 0) * (0.5 + 0.5 * lower_wick_atr)
        sweep_up_strength = df.get("liquidity_sweep_up", 0) * (0.5 + 0.5 * upper_wick_atr)

        df["ict_confluence_bull"] = (
            0.16 * fvg_bull_component
            + 0.20 * fresh_bull_ob
            + 0.10 * ob_strength_bull
            + 0.12 * df.get("in_ote_bullish", 0)
            + 0.08 * pd_align_bull
            + 0.13 * (df.get("displacement_confirmed_bull", df.get("displacement_bullish", 0)) * df.get("bos_up", 0))
            + 0.10 * sweep_down_strength
            + 0.06 * df.get("choch_bullish", 0)
            + 0.08 * df.get("inducement_bull", 0)
            + 0.05 * df.get("liquidity_raid_down", 0)
            + 0.04 * df.get("in_mitigation_zone_bull", 0)
        ).clip(0, 1.5)

        df["ict_confluence_bear"] = (
            0.16 * fvg_bear_component
            + 0.20 * fresh_bear_ob
            + 0.10 * ob_strength_bear
            + 0.12 * df.get("in_ote_bearish", 0)
            + 0.08 * pd_align_bear
            + 0.13 * (df.get("displacement_confirmed_bear", df.get("displacement_bearish", 0)) * df.get("bos_down", 0))
            + 0.10 * sweep_up_strength
            + 0.06 * df.get("choch_bearish", 0)
            + 0.08 * df.get("inducement_bear", 0)
            + 0.05 * df.get("liquidity_raid_up", 0)
            + 0.04 * df.get("in_mitigation_zone_bear", 0)
        ).clip(0, 1.5)

        df["ict_confluence_net"] = df["ict_confluence_bull"] - df["ict_confluence_bear"]
        df["ict_high_quality_bull"] = (df["ict_confluence_bull"] >= 0.6).astype(int)
        df["ict_high_quality_bear"] = (df["ict_confluence_bear"] >= 0.6).astype(int)
        return df

    def create_features(self, df, config, shift=True):
        df = self.add_sessions_and_killzones(df)
        df = self.compute_base_indicators(df)
        # df = self.add_gold_silver_ratio(df)  # Not applicable for USD/JPY
        df = self.add_swing_points(df, lookback=config.get("swing_lookback", 5))
        df = self.add_market_structure_ict(df)
        df = self.add_fair_value_gaps(df)
        df = self.add_order_blocks(df)
        df = self.add_liquidity(df)
        df = self.add_premium_discount(df)
        df = self.add_displacement(df)
        df = self.add_inducement(df)
        df = self.add_liquidity_void_raid(df)
        df = self.add_session_momentum(df)
        df = self.add_confluence_score(df)

        exclude_from_shift = [
            "datetime", "open", "high", "low", "close", "volume",
            "hour", "day_of_week", "hour_sin", "hour_cos",
            "is_asia", "is_london", "is_ny", "is_overlap", "is_friday",
            "kz_london_open", "kz_ny_am", "kz_london_close",
            "kz_silver_bullet_am", "kz_silver_bullet_pm",
        ]
        tech_cols = [c for c in df.columns if c not in exclude_from_shift]
        if shift:
            for col in tech_cols:
                df[col] = df[col].shift(1)

        thresh = int(len(df) * 0.5)
        df = df.dropna(axis=1, thresh=thresh)
        # الفلور الثابت (50 صف) كان مناسب للفريمات الصغيرة (آلاف الصفوف)
        # لكنه بيلتهم نسبة ضخمة من فريم زي 1W (~100-150 صف إجمالي فقط) —
        # وده السبب الأرجح وراء "0 صفقات" اللي ظهرت على 1W: بعد ما نشيل
        # 50 صف من أصل ~100-150، وبعدين نقسم holdout 15% وشريحة معايرة
        # كمان، عملياً مفيش بيانات كافية باقية. أغلب المؤشرات هنا محتاجة
        # نافذة أصغر بكتير من 50 (ATR=14، gold_silver_ratio بـ
        # min_periods=10)، فالفلور الآمن الحقيقي أقرب لـ 20-25 صف.
        warmup = max(20, len(df) // 20)
        df = df.iloc[warmup:].copy()
        df = df.ffill().fillna(0)

        exclude_final = ["datetime", "open", "high", "low", "close", "volume",
                          "future_close", "pct_change", "target"]
        self.feature_columns = [c for c in df.columns if c not in exclude_final]
        return df

    def create_target(self, df, horizon=1, min_atr_mult=None, threshold=None):
        df = df.copy()
        df["future_close"] = df["close"].shift(-horizon)
        df["pct_change"] = (df["future_close"] - df["close"]) / df["close"]
        move = df["future_close"] - df["close"]

        if threshold is not None:
            df["target"] = np.where(df["pct_change"] > threshold, 2,
                            np.where(df["pct_change"] < -threshold, 0, 1))
            df.dropna(subset=["future_close"], inplace=True)
            df["target"] = df["target"].astype(int)
            return df

        if min_atr_mult and "atr" in df.columns:
            move_thresh = min_atr_mult * df["atr"]
            raw_target = np.where(move > move_thresh, 1,
                          np.where(move < -move_thresh, 0, -1))
            df["target"] = raw_target
            n_before = len(df)
            df = df[df["target"] != -1].copy()
            n_after = len(df)
            logging.info(f"فلترة الهدف بـ ATR×{min_atr_mult}: اتشال {n_before - n_after} "
                          f"صف ضجيج من أصل {n_before} ({(n_before-n_after)/max(n_before,1)*100:.1f}%)")
        else:
            df["target"] = (df["future_close"] > df["close"]).astype(int)

        df.dropna(subset=["future_close"], inplace=True)
        df["target"] = df["target"].astype(int)
        return df

    def create_target_triple_barrier(self, df, horizon, atr_sl_mult, atr_tp_mult):
        df = df.copy()
        c = df["close"].values
        h = df["high"].values
        l = df["low"].values
        atr = df["atr"].values
        n = len(df)
        target = np.full(n, -1, dtype=int)

        for i in range(n - 1):
            a = atr[i]
            if np.isnan(a) or a <= 0:
                continue
            entry = c[i]
            tp = entry + atr_tp_mult * a
            sl = entry - atr_sl_mult * a
            end = min(i + 1 + horizon, n)
            for j in range(i + 1, end):
                if h[j] >= tp:
                    target[i] = 1
                    break
                if l[j] <= sl:
                    target[i] = 0
                    break

        df["target"] = target
        df["future_close"] = df["close"].shift(-horizon)
        df["pct_change"] = (df["future_close"] - df["close"]) / df["close"]
        n_before = len(df)
        df = df[df["target"] != -1].copy()
        n_after = len(df)
        logging.info(f"Triple-barrier target: اتشال {n_before - n_after} صف غامض "
                      f"(مفيش TP ولا SL اتحقق خلال {horizon} شمعة) من أصل {n_before} "
                      f"({(n_before-n_after)/max(n_before,1)*100:.1f}%) — الباقي حسم واضح")
        df["target"] = df["target"].astype(int)
        return df

    def fit_scaler(self, X):
        self.scaler.fit(X)
        self.scaler_fitted = True
        if isinstance(X, pd.DataFrame):
            self.feature_columns = list(X.columns)

    def transform(self, X):
        if not self.scaler_fitted:
            raise RuntimeError("الـ scaler لازم يتدرب أولاً (fit_scaler)")
        if isinstance(X, pd.DataFrame):
            X = X[self.feature_columns]
        return self.scaler.transform(X)

    def fit_transform(self, X):
        self.fit_scaler(X)
        return self.transform(X)


# ══════════════════════════════════════════════════════════════
# MULTI-TIMEFRAME BUILDER
# ══════════════════════════════════════════════════════════════
class MultiTimeframeBuilder:
    HTF_FEATURES = [
        "market_structure_trend", "trend_filter", "pd_position",
        "is_premium", "is_discount", "atr_pct", "bos_up", "bos_down",
        "choch_bullish", "choch_bearish",
    ]

    def __init__(self):
        self._htf_cache = {}

    def get_htf_features(self, htf_key):
        if htf_key in self._htf_cache:
            return self._htf_cache[htf_key]
        if htf_key not in TIMEFRAME_CONFIG:
            return None
        cfg = TIMEFRAME_CONFIG[htf_key]
        df_htf_raw = get_or_fetch_timeframe_data(htf_key, cfg)
        if df_htf_raw.empty:
            return None
        fe_htf = ICTFeatureEngineer()
        df_htf_feat = fe_htf.create_features(df_htf_raw, cfg, shift=True)
        self._htf_cache[htf_key] = df_htf_feat
        return df_htf_feat

    def merge_htf_bias(self, df_ltf, larger_tf_key, label):
        if not larger_tf_key:
            return df_ltf
        df_htf = self.get_htf_features(larger_tf_key)
        if df_htf is None:
            logging.warning(f"تعذر جلب HTF ({larger_tf_key}) لدمجه مع {label}")
            return df_ltf

        available = [c for c in self.HTF_FEATURES if c in df_htf.columns]
        df_htf_sel = df_htf[["datetime"] + available].copy()
        df_htf_sel.columns = ["datetime"] + [f"htf_{label}_{c}" for c in available]
        df_htf_sel = df_htf_sel.sort_values("datetime")

        df_merged = pd.merge_asof(
            df_ltf.sort_values("datetime"), df_htf_sel,
            on="datetime", direction="backward",
        )
        return df_merged.ffill().fillna(0)


def _wilson_ci(successes, n, z=1.96):
    if n == 0:
        return [0.0, 0.0]
    p = successes / n
    denom = 1 + z ** 2 / n
    center = (p + z ** 2 / (2 * n)) / denom
    margin = (z * ((p * (1 - p) / n + z ** 2 / (4 * n ** 2)) ** 0.5)) / denom
    return [round(max(0, (center - margin) * 100), 1), round(min(100, (center + margin) * 100), 1)]


# ══════════════════════════════════════════════════════════════
# BACKTESTER
# ══════════════════════════════════════════════════════════════
USDJPY_SPREAD_BPS = 1    # USD/JPY تاني أكثر زوج سيولة — سبريد ضيق
USDJPY_SLIPPAGE = 0.00003  # سليباج ضئيل جداً


class Backtester:
    def __init__(self, initial_capital=10000, spread_bps=USDJPY_SPREAD_BPS):
        self.initial_capital = initial_capital
        self.spread_bps = spread_bps / 10000
        self.slippage = USDJPY_SLIPPAGE

    def _exec_price(self, price, is_buy):
        if is_buy:
            return price * (1 + self.spread_bps) * (1 + self.slippage)
        return price * (1 - self.spread_bps) * (1 - self.slippage)

    def run(self, df, model, fe, feature_columns, config, use_fold_thresholds=False):
        capital, positions, trades = self.initial_capital, [], []
        missing = [c for c in feature_columns + ["close", "atr"] if c not in df.columns]
        if missing:
            logging.error(f"أعمدة مفقودة في الباك تيست: {missing}")
            return {}

        buy_th = config.get("fold_buy_threshold" if use_fold_thresholds else "buy_threshold")
        sell_th = config.get("fold_sell_threshold" if use_fold_thresholds else "sell_threshold")
        max_daily = config.get("max_trades_per_day")
        daily_counts = {}

        X_scaled = fe.transform(df[feature_columns].copy())
        all_proba = model.predict_proba(X_scaled)[:, 1]

        pct_above_buy = float((all_proba > buy_th).mean() * 100)
        pct_below_sell = float((all_proba < sell_th).mean() * 100)
        if pct_above_buy == 0 and pct_below_sell == 0:
            q = np.percentile(all_proba, [1, 25, 50, 75, 99])
            logging.debug(
                f"لا احتمال واحد عدّى buy_th={buy_th} أو نزل تحت sell_th={sell_th}. "
                f"توزيع الاحتمالات الفعلي — p1:{q[0]:.3f} p25:{q[1]:.3f} p50:{q[2]:.3f} "
                f"p75:{q[3]:.3f} p99:{q[4]:.3f}."
            )

        closes = df["close"].values
        atrs = df["atr"].values
        datetimes = df["datetime"].values if "datetime" in df.columns else None
        atr_sl = config.get("atr_sl_mult", 1.5)
        atr_tp = config.get("atr_tp_mult", 2.5)

        # ─── فلتر محاذاة HTF (اختياري لكل فريم عبر config) ───
        # فكرة ICT الأساسية "top-down analysis": الدخول لازم يتماشى مع
        # اتجاه الفريم الأعلى. من غير الفلتر ده، الصفقات المعاكسة
        # لاتجاه الـ HTF (short في ترند صاعد كبير، أو العكس) بتدخل
        # بحرية تامة وبتاكل من أي edge حقيقي — ده اللي ظهر بوضوح على
        # 4h/2h/1d بعد ما بقى الـ SHORT شغال: عدد الصفقات زاد لكن PF
        # نزل، يعني الصفقات الإضافية أغلبها معاكسة للاتجاه الأكبر.
        htf_trend_col = None
        if config.get("require_htf_alignment"):
            htf_trend_col = next(
                (c for c in df.columns if c.startswith("htf_") and c.endswith("_market_structure_trend")),
                None,
            )
        htf_trend = df[htf_trend_col].values if htf_trend_col else None

        for i in range(len(df)):
            p, atr, prob = closes[i], atrs[i], all_proba[i]
            if np.isnan(atr) or atr <= 0:
                continue

            if positions:
                pos = positions[0]
                exit_price, hit = None, None
                if pos["direction"] == "LONG":
                    if p <= pos["stop_loss"]:
                        exit_price, hit = pos["stop_loss"], "SL"
                    elif p >= pos["take_profit"]:
                        exit_price, hit = pos["take_profit"], "TP"
                else:
                    if p >= pos["stop_loss"]:
                        exit_price, hit = pos["stop_loss"], "SL"
                    elif p <= pos["take_profit"]:
                        exit_price, hit = pos["take_profit"], "TP"
                if exit_price is not None:
                    if pos["direction"] == "LONG":
                        pnl = (exit_price - pos["entry_price"]) * pos["units"]
                        capital += pos["units"] * exit_price
                    else:
                        pnl = (pos["entry_price"] - exit_price) * pos["units"]
                        capital -= pos["units"] * exit_price
                    trades.append({"pnl": pnl, "exit_reason": hit, "direction": pos["direction"]})
                    positions = []

            signal = "HOLD"
            if prob > buy_th:
                signal = "BUY"
            elif prob < sell_th:
                signal = "SELL"

            if htf_trend is not None and not np.isnan(htf_trend[i]):
                if signal == "BUY" and htf_trend[i] == -1:
                    signal = "HOLD"   # اتجاه HTF هابط — تجاهل إشارة الشراء
                elif signal == "SELL" and htf_trend[i] == 1:
                    signal = "HOLD"   # اتجاه HTF صاعد — تجاهل إشارة البيع

            if signal in ("BUY", "SELL") and max_daily and datetimes is not None:
                date_key = str(datetimes[i])[:10]
                if daily_counts.get(date_key, 0) >= max_daily:
                    signal = "HOLD"

            if signal == "BUY" and not positions:
                ep = self._exec_price(p, True)
                units = capital * 0.1 / ep
                positions.append({
                    "direction": "LONG",
                    "entry_price": ep, "units": units,
                    "stop_loss": ep - atr_sl * atr,
                    "take_profit": ep + atr_tp * atr,
                })
                capital -= units * ep
                if max_daily and datetimes is not None:
                    date_key = str(datetimes[i])[:10]
                    daily_counts[date_key] = daily_counts.get(date_key, 0) + 1
            elif signal == "SELL" and not positions:
                ep = self._exec_price(p, False)
                units = capital * 0.1 / ep
                positions.append({
                    "direction": "SHORT",
                    "entry_price": ep, "units": units,
                    "stop_loss": ep + atr_sl * atr,
                    "take_profit": ep - atr_tp * atr,
                })
                capital += units * ep
                if max_daily and datetimes is not None:
                    date_key = str(datetimes[i])[:10]
                    daily_counts[date_key] = daily_counts.get(date_key, 0) + 1

        if not trades:
            return {}

        pnls = np.array([t["pnl"] for t in trades])
        wins = pnls[pnls > 0]
        losses = pnls[pnls <= 0]
        wr = (len(wins) / len(pnls)) * 100 if len(pnls) else 0
        gross_profit = wins.sum() if len(wins) else 0
        gross_loss = abs(losses.sum()) if len(losses) else 0
        pf = gross_profit / gross_loss if gross_loss > 0 else (gross_profit if gross_profit > 0 else 0)
        equity_curve = self.initial_capital + np.cumsum(pnls)
        running_max = np.maximum.accumulate(equity_curve)
        dd = ((equity_curve - running_max) / running_max * 100).min() if len(equity_curve) else 0
        aw = wins.mean() if len(wins) else 0
        al = abs(losses.mean()) if len(losses) else 0

        return {
            "total_trades": len(trades),
            "win_rate": round(wr, 2),
            "win_rate_ci_95": _wilson_ci(len(wins), len(pnls)),
            "profit_factor": round(pf, 2),
            "total_return_pct": round((equity_curve[-1] - self.initial_capital) / self.initial_capital * 100, 2),
            "max_drawdown_pct": round(dd, 2),
            "avg_win": round(aw, 2),
            "avg_loss": round(al, 2),
            "expectancy": round((wr / 100 * aw) - ((1 - wr / 100) * al), 2) if al else 0,
            "statistically_reliable": len(trades) >= config.get("min_reliable_trades", 15),
        }


# ══════════════════════════════════════════════════════════════
# WALK-FORWARD TRAINING
# ══════════════════════════════════════════════════════════════
def expanding_walk_forward(X, y, df_full, config, n_splits=5, purge_bars=0):
    n = len(X)
    fold_size = n // (n_splits + 1)
    fold_metrics, best_model, best_fe, best_score = [], None, None, -np.inf
    fallback_model, fallback_fe, fallback_score = None, None, -np.inf
    best_n_trees, fallback_n_trees = None, None
    MIN_FOLD_TRADES = 3
    model = None

    for fold in range(n_splits):
        train_end = fold_size * (fold + 1)
        test_start = train_end + purge_bars
        test_end = min(test_start + fold_size, n)
        if test_start >= test_end:
            continue

        X_tr, y_tr = X.iloc[:train_end], y.iloc[:train_end]
        X_te, y_te = X.iloc[test_start:test_end], y.iloc[test_start:test_end]
        if len(X_tr) < 100 or len(X_te) < 20:
            continue

        neg, pos = (y_tr == 0).sum(), (y_tr == 1).sum()
        spw = neg / pos if pos > 0 else 1

        val_size = max(int(len(X_tr) * 0.15), 50)
        X_tr_fit, y_tr_fit = X_tr.iloc[:-val_size], y_tr.iloc[:-val_size]
        X_tr_val, y_tr_val = X_tr.iloc[-val_size:], y_tr.iloc[-val_size:]

        fe_fold = ICTFeatureEngineer()
        fe_fold.fit_scaler(X_tr_fit)
        X_tr_fit_s = fe_fold.transform(X_tr_fit)
        X_tr_val_s = fe_fold.transform(X_tr_val)
        X_te_s = fe_fold.transform(X_te)

        model = xgb.XGBClassifier(
            objective="binary:logistic", n_estimators=800, max_depth=4,
            learning_rate=0.03, subsample=0.7, colsample_bytree=0.6,
            reg_alpha=0.3, reg_lambda=2.5, min_child_weight=5, scale_pos_weight=spw,
            random_state=42, tree_method="hist", verbosity=0,
            eval_metric="logloss", early_stopping_rounds=40,
        )
        model.fit(X_tr_fit_s, y_tr_fit, eval_set=[(X_tr_val_s, y_tr_val)], verbose=False)

        n_trees_used = getattr(model, "best_iteration", None)
        logging.info(f"    Fold {fold+1}: عدد الأشجار المستخدمة فعلياً = "
                      f"{n_trees_used if n_trees_used is not None else model.n_estimators} "
                      f"(من أصل 800)")

        y_pred = model.predict(X_te_s)
        acc = accuracy_score(y_te, y_pred)
        prec = precision_score(y_te, y_pred, zero_division=0)
        rec = recall_score(y_te, y_pred, zero_division=0)

        df_test = df_full.iloc[test_start:test_end].copy()
        bt = Backtester().run(df_test, model, fe_fold, fe_fold.feature_columns,
                               config, use_fold_thresholds=True)

        used_dynamic_th = False
        if not bt or bt.get("total_trades", 0) == 0:
            X_te_scaled = fe_fold.transform(X_te)
            proba_te = model.predict_proba(X_te_scaled)[:, 1]
            dyn_buy = float(np.percentile(proba_te, 85))
            dyn_sell = float(np.percentile(proba_te, 15))
            dyn_config = dict(config)
            dyn_config["fold_buy_threshold"] = dyn_buy
            dyn_config["fold_sell_threshold"] = dyn_sell
            bt_dyn = Backtester().run(df_test, model, fe_fold, fe_fold.feature_columns,
                                       dyn_config, use_fold_thresholds=True)
            if bt_dyn and bt_dyn.get("total_trades", 0) > 0:
                bt = bt_dyn
                used_dynamic_th = True
                logging.info(f"    Fold {fold+1}: العتبة الثابتة مافيهاش صفقات → "
                              f"استخدمنا عتبة ديناميكية (buy>{dyn_buy:.3f}, sell<{dyn_sell:.3f}) "
                              f"للتقييم فقط")
            else:
                logging.warning(
                    f"    Fold {fold+1}: مفيش صفقات حتى بالعتبة الديناميكية (buy>{dyn_buy:.3f}, "
                    f"sell<{dyn_sell:.3f}) — الإشارة ضعيفة جداً في الفولد ده تحديداً"
                )

        logging.info(f"  Fold {fold+1}/{n_splits} — Acc:{acc:.2%} Prec:{prec:.2%} Rec:{rec:.2%}")
        if bt:
            logging.info(f"    BT{'(dyn)' if used_dynamic_th else ''} — WR:{bt['win_rate']}% "
                          f"PF:{bt['profit_factor']} Trades:{bt['total_trades']} "
                          f"Ret:{bt['total_return_pct']}%")

        fold_metrics.append({"fold": fold + 1, "accuracy": acc, "precision": prec,
                              "recall": rec, "backtest": bt,
                              "used_dynamic_threshold": used_dynamic_th})

        if bt and bt.get("total_trades", 0) > 0:
            score = bt["profit_factor"] * np.log1p(bt["total_trades"])
            if bt["total_trades"] >= MIN_FOLD_TRADES and score > best_score:
                best_score, best_model, best_fe = score, model, fe_fold
                best_n_trees = n_trees_used
            if score > fallback_score:
                fallback_score, fallback_model, fallback_fe = score, model, fe_fold
                fallback_n_trees = n_trees_used

    if best_model is None:
        if fallback_model is not None:
            logging.warning(
                f"لا فولد وصل لحد {MIN_FOLD_TRADES} صفقات — fallback لأفضل فولد "
                f"متاح (score={fallback_score:.3f}) بدل آخر فولد عشوائياً"
            )
            best_model, best_fe, best_n_trees = fallback_model, fallback_fe, fallback_n_trees
        elif model is not None:
            logging.warning("كل الفولدات بلا صفقات إطلاقاً — fallback لآخر موديل تدرب "
                             "(راجع الـ thresholds، ممكن تكون عالية جداً لحجم الداتا الحالي)")
            best_model, best_fe, best_n_trees = model, fe_fold, n_trees_used

    final_n_trees = best_n_trees
    if final_n_trees is None or final_n_trees < 10:
        final_n_trees = None
    return fold_metrics, best_model, best_fe, final_n_trees


def train_ict_model_for_timeframe(tf_key, n_splits=5, mtf_builder=None, force_refresh=False,
                                   target_mode=None):
    config = TIMEFRAME_CONFIG[tf_key]
    logging.info("=" * 60)
    logging.info(f"ICT Walk-Forward — فريم {tf_key}")
    logging.info("=" * 60)

    df_raw = get_or_fetch_timeframe_data(tf_key, config, force_refresh=force_refresh)
    logging.info(f"[{tf_key}] عدد الشموع المتاحة فعلياً: {len(df_raw)} "
                 f"(المطلوب كحد أدنى: {config['min_candles']})")
    if df_raw.empty or len(df_raw) < config["min_candles"]:
        logging.error(f"بيانات غير كافية لفريم {tf_key} ({len(df_raw)} شمعة)")
        return None

    fe = ICTFeatureEngineer()
    df_feat = fe.create_features(df_raw, config, shift=True)
    effective_mode = target_mode or ("barrier" if config.get("use_barrier_target", True) else "atr_filter")
    if effective_mode == "barrier":
        df_target = fe.create_target_triple_barrier(
            df_feat, horizon=config["horizon"],
            atr_sl_mult=config["atr_sl_mult"], atr_tp_mult=config["atr_tp_mult"],
        )
    else:
        df_target = fe.create_target(df_feat, horizon=config["horizon"],
                                      min_atr_mult=config.get("target_atr_mult"))
    logging.info(f"[{tf_key}] طريقة تصنيف الهدف المستخدمة: {effective_mode}")
    if df_target.empty:
        logging.error("لا بيانات بعد إنشاء الهدف")
        return None

    if mtf_builder and config.get("larger_tf"):
        df_target = mtf_builder.merge_htf_bias(df_target, config["larger_tf"], tf_key)

        htf_trend_col = f"htf_{tf_key}_market_structure_trend"
        if htf_trend_col in df_target.columns and "ict_confluence_bull" in df_target.columns:
            htf_trend = df_target[htf_trend_col]
            htf_bull_align = (htf_trend == 1).astype(int)
            htf_bear_align = (htf_trend == -1).astype(int)
            df_target["ict_confluence_bull_htf"] = df_target["ict_confluence_bull"] * (
                1.0 + 0.5 * htf_bull_align - 0.3 * htf_bear_align
            ).clip(0.2, 1.5)
            df_target["ict_confluence_bear_htf"] = df_target["ict_confluence_bear"] * (
                1.0 + 0.5 * htf_bear_align - 0.3 * htf_bull_align
            ).clip(0.2, 1.5)
            df_target["ict_confluence_net_htf"] = (
                df_target["ict_confluence_bull_htf"] - df_target["ict_confluence_bear_htf"]
            )

    exclude_final = ["datetime", "open", "high", "low", "close", "volume",
                      "future_close", "pct_change", "target"]
    fe.feature_columns = [c for c in df_target.columns if c not in exclude_final]
    df_target = df_target.reset_index(drop=True)

    # ─── تعديل تكيّفي على حجم الـ holdout ───
    # 15% ثابت كان بيعطي holdout صغير جداً على الفريمات العالية (4h/1d/1W)
    # — 14 صفقة أو أقل، اللي مش كافي إحصائياً لتقييم أي edge بشكل موثوق.
    # لو عندنا داتا كافية (على الأقل 800 صف قبل الـ holdout)، بنكبّر الـ
    # holdout لـ 20% عشان نضمن عدد صفقات أكبر في التقييم النهائي.
    adaptive_holdout_pct = HOLDOUT_PCT
    if len(df_target) >= 1000:
        adaptive_holdout_pct = 0.20
    holdout_start = int(len(df_target) * (1 - adaptive_holdout_pct))
    df_train_pool = df_target.iloc[:holdout_start].copy()
    df_holdout = df_target.iloc[holdout_start:].copy()
    logging.info(
        f"Train pool: {len(df_train_pool)} | Holdout ({int(adaptive_holdout_pct*100)}% - "
        f"لم يُرَ إطلاقاً أثناء التدريب): {len(df_holdout)}"
    )

    X_pool = df_train_pool[fe.feature_columns]
    y_pool = df_train_pool["target"]

    purge_bars = config["horizon"] * 2
    fold_metrics, best_model, best_fe, recommended_n_trees = expanding_walk_forward(
        X_pool, y_pool, df_train_pool, config, n_splits=n_splits, purge_bars=purge_bars
    )
    if best_model is None:
        logging.error(f"فشل التدريب لفريم {tf_key} — لا فولد صالح")
        return None
    n_trees_final = recommended_n_trees or best_model.n_estimators
    logging.info(f"عدد الأشجار المستخدم لموديلي التقييم/الإنتاج: {n_trees_final}")

    # ─── تقليل عدد الفيتشورز حسب الأهمية الفعلية (Feature Importance) قبل
    # مرحلة التقييم/المعايرة/الإنتاج — المشكلة اللي بتظهر فعلياً: عدد
    # الفيتشورز (~90-100) كبير جداً نسبة لحجم العينة المتاح (خصوصاً
    # الفريمات الأعلى زي 1d/1W اللي بيكون فيها بضع مئات صف بس)، وده نسبة
    # فيتشورز-لعينة عالية بتخلي أي موديل (حتى مع early stopping ورقابة)
    # عرضة قوية لحفظ نويز شريحة معينة بدل تعلّم نمط حقيقي معمّم — بالظبط
    # النمط اللي شفناه (عتبة تعدي المعايرة بس تفشل تماماً في الـ holdout).
    # الحل: ناخد ترتيب الأهمية الفعلي من موديل الـ CV (best_model، اتدرب
    # ووقف مبكراً على بيانات حقيقية Out-of-fold)، ونختار أفضل عدد محدود
    # منها بس (أقصى حد يعتمد على حجم العينة نفسها) لباقي المراحل.
    if best_model is not None and hasattr(best_model, "feature_importances_"):
        importances = best_model.feature_importances_
        feat_imp_pairs = sorted(zip(X_pool.columns, importances), key=lambda x: x[1], reverse=True)
        # قاعدة تقريبية: على الأقل ~8 صفوف لكل فيتشور، وسقف أقصى 45 فيتشور
        max_by_sample_size = max(15, len(X_pool) // 8)
        max_features = min(45, max_by_sample_size, len(feat_imp_pairs))
        selected_features = [f for f, imp in feat_imp_pairs[:max_features] if imp > 0]
        if len(selected_features) < 10:
            selected_features = [f for f, _ in feat_imp_pairs[:min(15, len(feat_imp_pairs))]]
        if len(selected_features) < len(X_pool.columns):
            logging.info(
                f"[{tf_key}] تقليل الفيتشورز من {len(X_pool.columns)} إلى "
                f"{len(selected_features)} (أهم فيتشورز بالـ Feature Importance من فولدات "
                f"الـ CV الحقيقية) — تقليل خطر الـ overfitting مع حجم العينة المتاح "
                f"({len(X_pool)} صف)."
            )
        fe.feature_columns = selected_features
        X_pool = X_pool[selected_features]
        y_pool = df_train_pool["target"]

    fe_eval = ICTFeatureEngineer()
    fe_eval.fit_scaler(X_pool)
    X_pool_s = fe_eval.transform(X_pool)
    neg, pos = (y_pool == 0).sum(), (y_pool == 1).sum()

    eval_model = xgb.XGBClassifier(
        objective="binary:logistic", n_estimators=n_trees_final,
        max_depth=best_model.max_depth, learning_rate=best_model.learning_rate,
        subsample=best_model.subsample, colsample_bytree=best_model.colsample_bytree,
        reg_alpha=best_model.reg_alpha, reg_lambda=best_model.reg_lambda,
        min_child_weight=best_model.min_child_weight,
        scale_pos_weight=neg / pos if pos else 1, random_state=42,
        tree_method="hist", verbosity=0, eval_metric="logloss",
    )
    eval_model.fit(X_pool_s, y_pool)

    # ─── شريحة المعايرة مقسومة لنصفين: "بحث" (الأقدم) و"تأكيد" (الأحدث) ───
    # المشكلة اللي بتحصل بدون الفحص ده: عتبة بتعدي بوابة الربحية على
    # شريحة معايرة واحدة (PF>1.05) لكن تطلع خسرانة تماماً على الـ holdout
    # الحقيقي بعدها — ده overfitting على نافذة زمنية واحدة بعينها، مش
    # edge حقيقي معمّم. الحل: قبل ما نقبل أي عتبة، لازم تثبت ربحيتها على
    # نصفين منفصلين زمنياً من شريحة المعايرة (نصف قديم للبحث + نصف أحدث
    # للتأكيد)، مش نافذة واحدة بس. ده مش بديل عن الـ holdout الحقيقي (لسه
    # معزول بالكامل ومايتلمسش هنا)، لكنه فلتر إضافي بيمسك overfitting قبل
    # ما نوصل للـ holdout أصلاً.
    calib_size = max(int(len(X_pool) * 0.15), 100)
    X_calib_train, y_calib_train = X_pool.iloc[:-calib_size], y_pool.iloc[:-calib_size]
    X_calib_test = X_pool.iloc[-calib_size:]
    df_calib_test = df_train_pool.iloc[-calib_size:]
    valid_atr_rows = int((df_calib_test["atr"].notna() & (df_calib_test["atr"] > 0)).sum())

    MIN_CONFIRM_ROWS = 30
    _half = len(X_calib_test) // 2
    X_calib_search, df_calib_search = X_calib_test.iloc[:_half], df_calib_test.iloc[:_half]
    X_calib_confirm, df_calib_confirm = X_calib_test.iloc[_half:], df_calib_test.iloc[_half:]
    has_confirm_slice = len(X_calib_search) >= MIN_CONFIRM_ROWS and len(X_calib_confirm) >= MIN_CONFIRM_ROWS
    logging.info(f"[{tf_key}] حجم شريحة المعايرة: {len(df_calib_test)} صف، منهم "
                 f"{valid_atr_rows} صف بـ ATR صالح (قابل فعلياً لفتح صفقة). "
                 f"إجمالي train_pool: {len(df_train_pool)}, إجمالي الداتا الخام "
                 f"بعد الفلترة: {len(df_target)}")

    calibrated_buy_th, calibrated_sell_th = config["buy_threshold"], config["sell_threshold"]
    calib_search_log = []
    no_viable_threshold = False
    if len(X_calib_train) <= 200 or len(X_calib_test) <= 50:
        logging.warning(
            f"[{tf_key}] بيانات قليلة جداً لتشغيل بحث المعايرة (calib_train="
            f"{len(X_calib_train)}, calib_test={len(X_calib_test)}) — هنستخدم عتبات "
            f"الكونفيج الثابتة مباشرة على الـ holdout بدون بحث. الحل الحقيقي هنا "
            f"محتاج تاريخ بيانات أطول لهذا الفريم."
        )
    if len(X_calib_train) > 200 and len(X_calib_test) > 50:
        cv_val_size = max(int(len(X_calib_train) * 0.15), 50)
        X_ct_fit, y_ct_fit = X_calib_train.iloc[:-cv_val_size], y_calib_train.iloc[:-cv_val_size]
        X_ct_val, y_ct_val = X_calib_train.iloc[-cv_val_size:], y_calib_train.iloc[-cv_val_size:]

        neg_c, pos_c = (y_ct_fit == 0).sum(), (y_ct_fit == 1).sum()
        fe_calib = ICTFeatureEngineer()
        fe_calib.fit_scaler(X_ct_fit)
        calib_model = xgb.XGBClassifier(
            objective="binary:logistic", n_estimators=800,
            max_depth=best_model.max_depth, learning_rate=best_model.learning_rate,
            subsample=best_model.subsample, colsample_bytree=best_model.colsample_bytree,
            reg_alpha=best_model.reg_alpha, reg_lambda=best_model.reg_lambda,
            min_child_weight=best_model.min_child_weight,
            scale_pos_weight=neg_c / pos_c if pos_c else 1, random_state=42,
            tree_method="hist", verbosity=0, eval_metric="logloss",
            early_stopping_rounds=40,
        )
        calib_model.fit(fe_calib.transform(X_ct_fit), y_ct_fit,
                         eval_set=[(fe_calib.transform(X_ct_val), y_ct_val)], verbose=False)
        calib_proba = calib_model.predict_proba(fe_calib.transform(X_calib_test))[:, 1]
        cq = np.percentile(calib_proba, [1, 25, 50, 75, 95, 99])
        logging.info(f"[{tf_key}] توزيع احتمالات المعايرة (n={len(calib_proba)}) — "
                      f"p1:{cq[0]:.3f} p25:{cq[1]:.3f} p50:{cq[2]:.3f} p75:{cq[3]:.3f} "
                      f"p95:{cq[4]:.3f} p99:{cq[5]:.3f}")

        # ─── رفع الحد الأدنى لعدد الصفقات المطلوبة عشان نعتبر أي PF رقم
        # ذو دلالة إحصائية فعلية. المشكلة اللي ظهرت فعلياً: عتبة بـ 3
        # صفقات بس بتظهر PF=2.92 في اللوج وكأنها مرشح حقيقي، والحقيقة إن
        # 3 صفقات مفيش فيها أي دلالة إحصائية (ممكن تبقى كلها صدفة). الحد
        # الأدنى القديم (5 أو حتى أقل) كان متساهل جداً. أي عتبة تحت الحد
        # الجديد بتتسجل في اللوج للشفافية لكن متتاخدش في الاعتبار كمرشح.
        min_trades_for_search = max(15, config.get("min_reliable_trades", 15) // 2)
        candidate_percentiles = [95, 92, 88, 84, 80, 76, 72]
        best_search_score = -np.inf
        if not has_confirm_slice:
            logging.info(
                f"[{tf_key}] شريحة المعايرة صغيرة جداً لتقسيمها لنصفين (بحث/تأكيد) — "
                f"هنكتفي بالبحث على الشريحة كلها زي الأول، من غير فحص تأكيد إضافي."
            )
        for pct in candidate_percentiles:
            cb = float(np.clip(np.percentile(calib_proba, pct), 0.502, 0.90))
            cs = float(np.clip(np.percentile(calib_proba, 100 - pct), 0.10, 0.498))
            if cb <= 0.5 or cs >= 0.5 or (cb - cs) <= 0.01:
                continue
            trial_config = dict(config)
            trial_config["buy_threshold"] = cb
            trial_config["sell_threshold"] = cs
            trial_bt = Backtester().run(df_calib_test, calib_model, fe_calib,
                                         fe_calib.feature_columns, trial_config,
                                         use_fold_thresholds=False)
            n_tr = trial_bt.get("total_trades", 0) if trial_bt else 0
            pf_tr = trial_bt.get("profit_factor", 0) if trial_bt else 0

            # ─── فحص التأكيد: هل نفس العتبة ربحية كمان على نص زمني تاني
            # منفصل تماماً عن النص اللي اتبحث فيه؟ لو لأ، دي عتبة اتفصّلت
            # على نافذة واحدة بعينها (overfitting)، مش edge حقيقي. ───
            confirm_pf, confirm_tr, search_pf, search_tr = None, 0, None, 0
            passes_confirm_split = True
            if has_confirm_slice:
                search_bt = Backtester().run(df_calib_search, calib_model, fe_calib,
                                              fe_calib.feature_columns, trial_config,
                                              use_fold_thresholds=False)
                confirm_bt = Backtester().run(df_calib_confirm, calib_model, fe_calib,
                                               fe_calib.feature_columns, trial_config,
                                               use_fold_thresholds=False)
                search_pf = search_bt.get("profit_factor", 0) if search_bt else 0
                search_tr = search_bt.get("total_trades", 0) if search_bt else 0
                confirm_pf = confirm_bt.get("profit_factor", 0) if confirm_bt else 0
                confirm_tr = confirm_bt.get("total_trades", 0) if confirm_bt else 0
                min_half_trades = max(3, min_trades_for_search // 2)
                passes_confirm_split = (
                    search_tr >= min_half_trades and confirm_tr >= min_half_trades and
                    search_pf > 1.0 and confirm_pf > 1.0
                )

            insufficient_sample = n_tr < min_trades_for_search
            calib_search_log.append({
                "percentile": pct, "buy_th": round(cb, 3), "sell_th": round(cs, 3),
                "trades": n_tr, "profit_factor": pf_tr,
                "confirm_split_search_pf": round(search_pf, 2) if search_pf is not None else None,
                "confirm_split_confirm_pf": round(confirm_pf, 2) if confirm_pf is not None else None,
                "passes_confirm_split": passes_confirm_split,
                "insufficient_sample": insufficient_sample,
            })
            if n_tr >= min_trades_for_search and pf_tr > 1.05 and passes_confirm_split:
                score = pf_tr * np.log1p(n_tr)
                if score > best_search_score:
                    best_search_score = score
                    calibrated_buy_th, calibrated_sell_th = cb, cs

        if best_search_score > -np.inf:
            search_summary = ", ".join(
                f"{c['percentile']}%→PF{c['profit_factor']}@{c['trades']}تج"
                + (" ⚠️عينة_صغيرة_متجاهلة" if c.get("insufficient_sample") else "")
                + (f"[تأكيد:بحث{c['confirm_split_search_pf']}/تأكيد{c['confirm_split_confirm_pf']}"
                   f"{'✅' if c['passes_confirm_split'] else '❌'}]"
                   if c.get("confirm_split_search_pf") is not None else "")
                for c in calib_search_log
            )
            logging.info(f"✅ عتبات معايرة عبر بحث فعلي على أداء حقيقي — "
                          f"buy={calibrated_buy_th:.3f} sell={calibrated_sell_th:.3f} "
                          f"(score={best_search_score:.3f}) — [{search_summary}]")
        else:
            static_trial = dict(config)
            static_bt = Backtester().run(df_calib_test, calib_model, fe_calib,
                                          fe_calib.feature_columns, static_trial,
                                          use_fold_thresholds=False)
            static_pf = static_bt.get("profit_factor", 0) if static_bt else 0
            static_trades = static_bt.get("total_trades", 0) if static_bt else 0
            if static_pf > 1.05 and static_trades >= min_trades_for_search:
                logging.warning(
                    f"[{tf_key}] البحث مالقاش عتبة أفضل من الثابتة — هنستخدم "
                    f"القيم الثابتة في الكونفيج (PF={static_pf} على شريحة المعايرة)"
                )
            else:
                no_viable_threshold = True
                search_summary2 = ", ".join(
                    f"{c['percentile']}%→PF{c['profit_factor']}@{c['trades']}تج"
                    + (" ⚠️عينة_صغيرة_متجاهلة" if c.get("insufficient_sample") else "")
                    for c in calib_search_log
                )
                logging.warning(
                    f"[{tf_key}] ⚠️ لا توجد عتبة (لا من البحث ولا الثابتة) بتحقق "
                    f"ربحية حقيقية (PF>1.05) على شريحة المعايرة — [{search_summary2}, "
                    f"ثابتة→PF{static_pf}@{static_trades}تج]. الفريم ده مفيهوش edge "
                    f"حقيقي حالياً كاستراتيجية مستقلة بالفيتشورز الحالية — استخدمه "
                    f"كـ HTF context/bias بس، مش كقرار تداول قائم بذاته."
                )

    calibrated_config = dict(config)
    calibrated_config["buy_threshold"] = calibrated_buy_th
    calibrated_config["sell_threshold"] = calibrated_sell_th
    calibrated_config["viable_for_standalone_trading"] = not no_viable_threshold

    holdout_bt = Backtester().run(
        df_holdout, eval_model, fe_eval, fe_eval.feature_columns, calibrated_config,
        use_fold_thresholds=False,
    )
    if holdout_bt:
        reliability_tag = "✅ موثوق إحصائياً" if holdout_bt["statistically_reliable"] else \
            f"⚠️ غير موثوق (أقل من {config.get('min_reliable_trades', 15)} صفقة)"
        logging.info(f"{'✅' if holdout_bt['statistically_reliable'] else '⚠️'} "
                      f"Holdout حقيقي (لا تسريب) — WR:{holdout_bt['win_rate']}% "
                      f"(CI95: {holdout_bt['win_rate_ci_95'][0]}-{holdout_bt['win_rate_ci_95'][1]}%) "
                      f"PF:{holdout_bt['profit_factor']} Trades:{holdout_bt['total_trades']} "
                      f"Ret:{holdout_bt['total_return_pct']}% DD:{holdout_bt['max_drawdown_pct']}% "
                      f"— {reliability_tag}")
        if not holdout_bt["statistically_reliable"]:
            logging.warning(
                f"[{tf_key}] عدد صفقات الـ holdout ({holdout_bt['total_trades']}) قليل جداً — "
                f"أي رقم PF أو Win Rate هنا (حتى لو شكله ممتاز زي PF عالي جداً) مش دليل "
                f"موثوق على أداء حقيقي. محتاج فترة تاريخية أطول أو تكرار الاختبار على "
                f"فترات مختلفة قبل ما نثق في الأرقام دي."
            )
    else:
        logging.warning(f"لا صفقات في الـ holdout لفريم {tf_key} حتى بعد المعايرة — "
                         f"الإشارة المتاحة ضعيفة جداً لهذا الفريم بالفيتشورز الحالية")

    fold_pfs = [
        min(fm["backtest"].get("profit_factor", 0), 10)
        for fm in fold_metrics if fm.get("backtest")
    ]
    if len(fold_pfs) >= 2:
        pf_median = float(np.median(fold_pfs))
        profitable_fold_ratio = sum(1 for p in fold_pfs if p > 1.0) / len(fold_pfs)
        consistent = profitable_fold_ratio >= 0.66 and pf_median > 1.1
        logging.info(
            f"{'✅' if consistent else '⚠️'} [{tf_key}] استقرار عبر {len(fold_pfs)} فولدات "
            f"CV (كلها out-of-sample حقيقياً) — PF لكل فولد (مقصوصة عند 10): "
            f"{[round(p,2) for p in fold_pfs]} (وسيط={pf_median:.2f}, فولدات رابحة: "
            f"{sum(1 for p in fold_pfs if p > 1.0)}/{len(fold_pfs)}) — "
            f"{'edge متسق نسبياً عبر فترات مختلفة' if consistent else 'الأداء متذبذب — مؤشر قوي على عدم وجود edge حقيقي مستقر، مش بس مشكلة عتبة أو معايرة'}"
        )
        calibrated_config["stability_check_pf_by_fold"] = [round(p, 2) for p in fold_pfs]
        calibrated_config["stability_consistent"] = consistent

    os.makedirs(MODELS_DIR, exist_ok=True)

    X_full = df_target[fe.feature_columns]
    y_full = df_target["target"]
    fe_prod = ICTFeatureEngineer()
    fe_prod.fit_scaler(X_full)
    X_full_s = fe_prod.transform(X_full)
    neg_f, pos_f = (y_full == 0).sum(), (y_full == 1).sum()

    prod_model = xgb.XGBClassifier(
        objective="binary:logistic", n_estimators=n_trees_final,
        max_depth=best_model.max_depth, learning_rate=best_model.learning_rate,
        subsample=best_model.subsample, colsample_bytree=best_model.colsample_bytree,
        reg_alpha=best_model.reg_alpha, reg_lambda=best_model.reg_lambda,
        min_child_weight=best_model.min_child_weight,
        scale_pos_weight=neg_f / pos_f if pos_f else 1, random_state=42,
        tree_method="hist", verbosity=0, eval_metric="logloss",
    )
    prod_model.fit(X_full_s, y_full)

    importances = prod_model.feature_importances_
    feat_imp_pairs = sorted(zip(fe_prod.feature_columns, importances),
                             key=lambda x: x[1], reverse=True)
    top_20 = feat_imp_pairs[:20]
    ict_keywords = ("ob_", "fvg", "confluence", "inducement", "liquidity", "breaker",
                     "mitigat", "displacement", "ote", "choch", "bos_", "premium",
                     "discount", "session_momentum", "killzone", "kz_")
    ict_importance_share = sum(
        imp for feat, imp in feat_imp_pairs
        if any(k in feat.lower() for k in ict_keywords)
    )
    logging.info(f"[{tf_key}] Top 10 Feature Importance: " +
                 ", ".join(f"{f}={round(float(i),3)}" for f, i in top_20[:10]))
    logging.info(f"[{tf_key}] نسبة أهمية فيتشورز ICT من الإجمالي: "
                 f"{ict_importance_share*100:.1f}%")

    model_path = os.path.join(MODELS_DIR, f"{SYMBOL_SAFE}_{tf_key}_ict_usdjpy.pkl")
    joblib.dump({
        "model": prod_model, "scaler": fe_prod.scaler,
        "feature_columns": fe_prod.feature_columns, "config": calibrated_config,
        "note": "trained on 100% of data (incl. holdout) — for live deployment only, "
                "holdout metrics below do NOT apply to this exact model. "
                "buy_threshold/sell_threshold in this config are auto-calibrated "
                "from a held-out calibration slice, not fixed guesses.",
    }, model_path)

    metrics_path = os.path.join(MODELS_DIR, f"{SYMBOL_SAFE}_{tf_key}_ict_usdjpy_metrics.json")
    with open(metrics_path, "w") as f:
        json.dump({
            "timeframe": tf_key,
            "fold_results": fold_metrics,
            "holdout_backtest_true_oos": holdout_bt,
            "holdout_note": "evaluated with eval_model, trained ONLY on the pre-holdout "
                             "train_pool — never saw holdout data. This is the honest metric.",
            "config_static": config,
            "config_calibrated": calibrated_config,
            "n_features": len(fe.feature_columns),
            "top_20_feature_importance": [
                {"feature": f, "importance": round(float(i), 4)} for f, i in top_20
            ],
            "ict_features_importance_share_pct": round(ict_importance_share * 100, 1),
        }, f, indent=2, default=str)

    logging.info(f"تم حفظ نموذج الإنتاج: {model_path}")
    viable = calibrated_config.get("viable_for_standalone_trading", True)
    return prod_model, fe_prod, holdout_bt, viable, effective_mode


def _holdout_truly_profitable(holdout_bt, min_pf=1.05):
    """
    فحص متدرج للأداء على شريحة الـ holdout (بيانات لم تُرَ إطلاقاً أثناء
    التدريب أو المعايرة). المشكلة اللي كانت في النسخة القديمة: كانت
    بتستخدم عتبة قاطعة (PF > 1.05) — أي holdout بـ PF=1.05 بالظبط أو
    1.04 كان بيتعامل كـ "فشل تام"، رغم إن ده أداء مقبول لموديل حقيقي
    (خصوصاً مع عدد صفقات محترم على holdout صغير). ده كان بيرفض موديلات
    فيها edge حقيقي بس ضعيف، ويصنّفها كـ overfitting، وهو مش دقيق.

    النسخة الجديدة: قبول متدرج بيراعي عدد الصفقات:
      - PF >= 1.15 + عدد صفقات >= 10 → قبول واضح (edge معقول)
      - PF >= 1.05 + عدد صفقات >= 20 → قبول (edge ضعيف لكن مستقر إحصائياً)
      - PF >= 1.00 + عدد صفقات >= 40 → قبول حذر (لا خسارة + عينة كبيرة)
      - غير كده → رفض (مش موثوق للتداول المستقل)
    ده منطق أقرب للواقع: عينة كبيرة بـ PF متواضع أدل على edge حقيقي من
    عينة صغيرة بـ PF عالي (اللي ممكن يكون صدفة).
    """
    if not holdout_bt:
        return False
    pf = holdout_bt.get("profit_factor", 0)
    n = holdout_bt.get("total_trades", 0)
    if pf >= 1.15 and n >= 10:
        return True
    if pf >= 1.05 and n >= 20:
        return True
    if pf >= 1.00 and n >= 40:
        return True
    return False


def train_ict_model_auto(tf_key, n_splits=5, mtf_builder=None, force_refresh=False):
    result = train_ict_model_for_timeframe(
        tf_key, n_splits=n_splits, mtf_builder=mtf_builder,
        force_refresh=force_refresh, target_mode="barrier",
    )
    if result is None:
        return None
    prod_model, fe_prod, holdout_bt, viable, mode_used = result
    holdout_ok = _holdout_truly_profitable(holdout_bt)
    if viable and holdout_ok:
        return prod_model, fe_prod, holdout_bt

    if viable and not holdout_ok:
        _pf = holdout_bt.get('profit_factor', 0) if holdout_bt else 0
        _n = holdout_bt.get('total_trades', 0) if holdout_bt else 0
        logging.warning(
            f"[{tf_key}] ⚠️ triple-barrier عدّى بوابة المعايرة (viable=True) لكن "
            f"الـ holdout الحقيقي طلع PF={_pf} على {_n} صفقة — مش كافي لبوابة "
            f"القبول المتدرجة (محتاجين PF≥1.15 مع 10+ صفقات، أو PF≥1.05 مع 20+، "
            f"أو PF≥1.00 مع 40+). جاري تجربة الطريقة البديلة (atr_filter)..."
        )
    else:
        logging.warning(f"[{tf_key}] طريقة triple-barrier مالقتش عتبة مربحة — "
                         f"جاري إعادة المحاولة بطريقة فلترة ATR البسيطة كبديل...")

    result2 = train_ict_model_for_timeframe(
        tf_key, n_splits=n_splits, mtf_builder=mtf_builder,
        force_refresh=False, target_mode="atr_filter",
    )
    if result2 is None:
        logging.warning(f"[{tf_key}] الطريقة البديلة فشلت كمان — هنفضل بموديل "
                         f"triple-barrier (غير مضمون الربحية، لكن الوحيد المتاح)")
        return prod_model, fe_prod, holdout_bt

    prod_model2, fe_prod2, holdout_bt2, viable2, mode_used2 = result2
    holdout2_ok = _holdout_truly_profitable(holdout_bt2)

    if viable2 and holdout2_ok:
        logging.info(f"✅ [{tf_key}] الطريقة البديلة (atr_filter) عدّت المعايرة "
                      f"*و* الـ holdout الحقيقي — هنستخدمها")
        return prod_model2, fe_prod2, holdout_bt2

    # ─── ولا طريقة عمّمت فعلياً على الـ holdout — نختار الأفضل بينهم
    # بنفس مقياس التقييم المستخدم داخلياً (PF × log(1+trades))، ونعلّم
    # النتيجة بوضوح إنها غير موثوقة للتداول المستقل، بدل ما نتظاهر إن
    # أي منهم "نجح" لمجرد إنه عدّى بوابة المعايرة. ───
    def _score(bt):
        if not bt or not bt.get("total_trades"):
            return -np.inf
        return bt.get("profit_factor", 0) * np.log1p(bt["total_trades"])

    score1, score2 = _score(holdout_bt), _score(holdout_bt2)
    if score2 > score1:
        logging.warning(
            f"[{tf_key}] ولا طريقة عمّمت فعلياً للـ holdout — atr_filter أفضل قليلاً "
            f"(holdout PF={holdout_bt2.get('profit_factor', 0) if holdout_bt2 else 0}) "
            f"من triple-barrier (PF={holdout_bt.get('profit_factor', 0) if holdout_bt else 0}). "
            f"هنحفظه، لكنه *غير موثوق* للتداول المستقل حالياً — استخدمه كـ HTF "
            f"context/bias بس لحد ما يبان edge أوضح ببيانات/فيتشورز إضافية."
        )
        return prod_model2, fe_prod2, holdout_bt2
    else:
        logging.warning(
            f"[{tf_key}] ولا الطريقتين عمّمتا فعلياً للـ holdout — هنفضل بـ "
            f"triple-barrier (PF={holdout_bt.get('profit_factor', 0) if holdout_bt else 0}) "
            f"كأفضل الموجود، لكنه *غير موثوق* للتداول المستقل حالياً."
        )
        return prod_model, fe_prod, holdout_bt


def train_all_ict_models(force_refresh_data=False):
    logging.info("=" * 70)
    logging.info("ICT USD/JPY Model Factory")
    logging.info("=" * 70)

    mtf_builder = MultiTimeframeBuilder()
    results = {}
    for tf in TIMEFRAME_CONFIG:
        res = train_ict_model_auto(
            tf, n_splits=5, mtf_builder=mtf_builder, force_refresh=force_refresh_data
        )
        results[tf] = "✅" if res else "❌"

    logging.info("=" * 70)
    logging.info("ملخص التدريب:")
    for tf, st in results.items():
        logging.info(f"   {tf}: {st}")
    logging.info("=" * 70)


if __name__ == "__main__":
    train_all_ict_models(force_refresh_data=False)

## تحميل الموديلات
شغّل الخلية دي بعد انتهاء التدريب فوق عشان تنزّل كل الموديلات كملف ZIP.

In [3]:
# ============================================================
# download_models
import os
import shutil
from google.colab import files

MODELS_DIR = "models/ict_usdjpy"
ZIP_NAME   = "ict_usdjpy.zip"

if not os.path.exists(MODELS_DIR):
    print(f"❌ الفولدر '{MODELS_DIR}' مش موجود!")
    print("تأكد إن الخلية اللي فوق اشتغلت وحفظت الملفات أولاً.")
else:
    all_files = os.listdir(MODELS_DIR)
    pkl_files  = [f for f in all_files if f.endswith('.pkl')]
    json_files = [f for f in all_files if f.endswith('.json')]

    print(f"📁 محتوى '{MODELS_DIR}':")
    print(f"   .pkl  (موديلات): {len(pkl_files)}")
    print(f"   .json (مقاييس):  {len(json_files)}")
    print()
    for f in sorted(all_files):
        size = os.path.getsize(os.path.join(MODELS_DIR, f))
        print(f"   {f:<45} {size/1024:.1f} KB")

    print(f"\n📦 جاري ضغط الفولدر ...")
    shutil.make_archive(ZIP_NAME.replace('.zip', ''), 'zip', '.', MODELS_DIR)

    size_mb = os.path.getsize(ZIP_NAME) / (1024 * 1024)
    print(f"✅ تم إنشاء: {ZIP_NAME} ({size_mb:.1f} MB)")

    print(f"⬇️  جاري التحميل ...")
    files.download(ZIP_NAME)
    print(f"✅ تم!")


📁 محتوى 'models/ict_usdjpy':
   .pkl  (موديلات): 8
   .json (مقاييس):  8

   USDJPY_15min_ict_usdjpy.pkl                   81.1 KB
   USDJPY_15min_ict_usdjpy_metrics.json          5.9 KB
   USDJPY_1W_ict_usdjpy.pkl                      923.6 KB
   USDJPY_1W_ict_usdjpy_metrics.json             5.8 KB
   USDJPY_1d_ict_usdjpy.pkl                      1061.6 KB
   USDJPY_1d_ict_usdjpy_metrics.json             5.8 KB
   USDJPY_1h_ict_usdjpy.pkl                      77.2 KB
   USDJPY_1h_ict_usdjpy_metrics.json             5.5 KB
   USDJPY_2h_ict_usdjpy.pkl                      44.0 KB
   USDJPY_2h_ict_usdjpy_metrics.json             5.9 KB
   USDJPY_45min_ict_usdjpy.pkl                   420.0 KB
   USDJPY_45min_ict_usdjpy_metrics.json          5.9 KB
   USDJPY_4h_ict_usdjpy.pkl                      1194.0 KB
   USDJPY_4h_ict_usdjpy_metrics.json             6.0 KB
   USDJPY_5min_ict_usdjpy.pkl                    144.6 KB
   USDJPY_5min_ict_usdjpy_metrics.json           5.9 KB

📦 جاري ضغط الف

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ تم!
